In [1]:
import os
import math
import random
from typing import Tuple, List, Dict
import copy
import time
import psutil

import inspect


import h5py
import joblib


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils import weight_norm

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score, classification_report

import tensorflow as tf


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

2026-02-02 17:20:44.844590: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770052845.218577     164 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770052845.335230     164 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770052846.327122     164 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770052846.327160     164 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770052846.327163     164 computation_placer.cc:177] computation placer alr

Device: cuda


In [2]:
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

## Configurations

In [3]:
# file_path = f'/kaggle/input/fypdata/combined_matches_2068 2269 2417.csv'
file_path = f'/kaggle/input/fypdata/combined_matches.csv'

USE_PREPROCESSED = False
PREPROCESSED_DIR = '/kaggle/input/preprocessed-sequences/'  # Change path if needed for Kaggle: '/kaggle/input/preprocessed-sequences/'



N_ROWS = 8
N_COLS = 18
NUM_ZONES = N_ROWS * N_COLS

# Sequence & horizon
SEQ_LEN =100        # number of past frames used
FPS = 25            # adjust based on your data (often 25fps for football)
HORIZON_SECONDS = 2 # predict location after 3/5 seconds
HORIZON_FRAMES = HORIZON_SECONDS * FPS

# Training params 
BATCH_SIZE =16
LR = 1e-4
EPOCHS = 20
PATIENCE = 2  # for early stopping
CO_ORDINATES= True
# Enhanced features for better zone prediction
FEATURE_COLS = [
    # Position
    "x_normalized", "y_normalized",
    
    # Basic movement
    "dx", "dy", "speed_normalized", "acceleration", "movement_angle",
    
    # Multi-window velocities (NEW)
    "dx_avg_3", "dy_avg_3", "dx_avg_5", "dy_avg_5", "dx_avg_10", "dy_avg_10",
    "speed_avg_3", "speed_avg_5", "speed_avg_10",
    
    # Movement trends (NEW)
    "acceleration_trend", "angle_change", "angle_stability", "speed_change_rate",
    
    # Spatial context
    "distance_from_center", "distance_from_goal_home", "distance_from_goal_away",
    "distance_from_sideline",
    
    # Ball & team
    "distance_to_ball", "ball_possession_proximity", "team_has_possession", "team_spread"
]


coord_cols=("x_normalized", "y_normalized")

some_number= 42

worker_num=4

MODEL_TYPE="Keras_tcn" #tcn/enhanced/tcn_new/LocusLab_tcn/Keras_tcn
BACKEND = "keras" #keras/torch

print(f"Using {len(FEATURE_COLS)} features for prediction")
print(f"Grid configuration: {N_ROWS}x{N_COLS} = {NUM_ZONES} zones")
print(f"Prediction horizon: {HORIZON_SECONDS} seconds ({HORIZON_FRAMES} frames)")
print(f"Using framework:{BACKEND}")
print(f"Using Model:{MODEL_TYPE}")
print(f"Using framework:{BACKEND}")
print(f"Using batch size: {BATCH_SIZE}: {BATCH_SIZE}")


Using 28 features for prediction
Grid configuration: 8x18 = 144 zones
Prediction horizon: 2 seconds (50 frames)
Using framework:keras
Using Model:Keras_tcn
Using framework:keras
Using batch size: 16: 16


In [4]:
def xy_to_zone_vectorized(xs, ys, n_rows=N_ROWS, n_cols=N_COLS, flip_y=False):
    xs = np.asarray(xs)
    ys = np.asarray(ys)

    if flip_y:
        ys = 1 - ys

    row = np.clip((ys * n_rows).astype(int), 0, n_rows - 1)
    col = np.clip((xs * n_cols).astype(int), 0, n_cols - 1)
    return row * n_cols + col


In [5]:

def split_by_match_df(df, train_frac=0.7, val_frac=0.15, seed=42):
    """
    Splits a dataframe into train/val/test by unique match_id.
    Ensures no leakage across matches.
    """
    # get unique match_ids and shuffle
    unique_matches = df["match_id"].unique()
    np.random.seed(seed)
    np.random.shuffle(unique_matches)

    n_total = len(unique_matches)
    n_train = max(1, int(train_frac * n_total))
    n_val   = max(1, int(val_frac * n_total))
    n_train = min(n_train, n_total - n_val)  # ensure enough for test

    train_matches = unique_matches[:n_train]
    val_matches   = unique_matches[n_train:n_train+n_val]
    test_matches  = unique_matches[n_train+n_val:]

    train_df = df[df["match_id"].isin(train_matches)].reset_index(drop=True)
    val_df   = df[df["match_id"].isin(val_matches)].reset_index(drop=True)
    test_df  = df[df["match_id"].isin(test_matches)].reset_index(drop=True)

    print("Split matches:", len(train_matches), len(val_matches), len(test_matches))
    print("Rows per split:", len(train_df), len(val_df), len(test_df))

    return train_df, val_df, test_df


## Football Sequence Dataset class & pytorch 

In [6]:
class FootballSequenceDataset(Dataset):
    def __init__(self, df, seq_len=SEQ_LEN, horizon=HORIZON_FRAMES,
                 flip_y=False, feature_cols=FEATURE_COLS, scaler=None):
        self.seq_len = seq_len
        self.horizon = horizon
        self.feature_cols = feature_cols
        self.flip_y = flip_y
        self.scaler = scaler

        # store groups (player, match level)
        self.groups = []
        grouped = df.groupby(["match_id", "player_id"])

        # coords = g[list(coord_cols)].values.astype(np.float32)
        
        for (match, pid), g in grouped:
            g = g.reset_index(drop=True)
            if len(g) < seq_len + horizon:
                continue

            #co_ordinates
            coords = g[list(coord_cols)].values.astype(np.float32)

            if flip_y:
                coords[:, 1] = 1.0 - coords[:, 1]
                
            # features
            feats = g[feature_cols].values.astype(np.float32)
            
            if self.scaler is not None:
                feats = self.scaler.transform(feats)

            # precompute zones
            zones = xy_to_zone_vectorized(
                g["x_normalized"].values,
                g["y_normalized"].values,
                N_ROWS, N_COLS,
                flip_y=False
            )

            self.groups.append({
                "match": match,
                "player": pid,
                "features": feats,
                "zones": zones,
                "coords": coords
            })

        # index mapping (dataset idx → group, start_idx)
        self.index_map = []
        for gi, group in enumerate(self.groups):
            n = len(group["features"])
            valid_starts = n - (seq_len + horizon) + 1
            for start in range(valid_starts):
                self.index_map.append((gi, start))

    def __len__(self):
        return len(self.index_map)

    def __getitem__(self, idx):
        gi, start = self.index_map[idx]
        group = self.groups[gi]
        
        # Input sequence        
        seq = group["features"][start:start+self.seq_len]
        label = group["zones"][start+self.seq_len-1 + self.horizon]
        

        # Indices
        t_curr = start + self.seq_len - 1
        t_fut = t_curr + self.horizon

        targets= {}

        X_COL = self.feature_cols.index("x_normalized")
        Y_COL = self.feature_cols.index("y_normalized")

        if (CO_ORDINATES):
            curr_xy = group["coords"][t_curr]
            fut_xy = group["coords"][t_fut]
            delta = fut_xy - curr_xy
            targets["delta"] = torch.tensor(delta, dtype=torch.float32)
        else:
            zone = group["zones"][t_fut]
            targets["zone"] = torch.tensor(zone, dtype=torch.long)

        

        return torch.tensor(seq, dtype=torch.float32), targets

## Keras Sequence & Keras

In [7]:
def build_keras_sequences(df, feature_cols, scaler, seq_len, horizon, coordinate_targets, n_rows=None, n_cols=None):
    X, y = [], []

    for _, player_df in df.groupby(["match_id", "player_id"]):
        feats = player_df[feature_cols].values.astype(np.float32)
        if scaler is not None:
            feats = scaler.transform(feats)

        coords = player_df[list(coord_cols)].values.astype(np.float32)

        if coordinate_targets:
            # regression: dx, dy
            for i in range(len(player_df) - seq_len - horizon + 1):
                seq = feats[i:i+seq_len]
                t_curr = i + seq_len - 1
                t_fut = t_curr + horizon
                delta = coords[t_fut] - coords[t_curr]
                X.append(seq)
                y.append(delta)
        else:
            # classification: zones
            zones = xy_to_zone_vectorized(
                player_df["x_normalized"].values,
                player_df["y_normalized"].values,
                n_rows, n_cols
            )
            for i in range(len(player_df) - seq_len - horizon + 1):
                seq = feats[i:i+seq_len]
                t_fut = i + seq_len - 1 + horizon
                zone = zones[t_fut]
                X.append(seq)
                y.append(zone)

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32 if coordinate_targets else np.int64)
    return X, y


In [8]:
def keras_sequence_generator(df, feature_cols, scaler, seq_len, horizon,
                             coordinate_targets, n_rows=None, n_cols=None):
    """
    Generator that yields sequences one by one (or in small batches) instead of storing all.
    """
    for _, player_df in df.groupby(["match_id", "player_id"]):
        feats = player_df[feature_cols].values.astype(np.float32)
        if scaler is not None:
            feats = scaler.transform(feats)
        coords = player_df[list(coord_cols)].values.astype(np.float32)

        if coordinate_targets:
            # regression: dx, dy
            for i in range(len(player_df) - seq_len - horizon + 1):
                seq = feats[i:i+seq_len]
                t_curr = i + seq_len - 1
                t_fut = t_curr + horizon
                delta = coords[t_fut] - coords[t_curr]
                yield seq, delta.astype(np.float32)
        else:
            zones = xy_to_zone_vectorized(
                player_df["x_normalized"].values,
                player_df["y_normalized"].values,
                n_rows, n_cols
            )
            for i in range(len(player_df) - seq_len - horizon + 1):
                seq = feats[i:i+seq_len]
                t_fut = i + seq_len - 1 + horizon
                zone = zones[t_fut]
                yield seq, np.int32(zone)


In [9]:
def make_tf_dataset(df, batch_size, shuffle=False, coordinate_targets=False):
    output_types = (tf.float32, tf.float32 if coordinate_targets else tf.int32)
    output_shapes = ((SEQ_LEN, len(FEATURE_COLS)), (2,) if coordinate_targets else ())

    ds = tf.data.Dataset.from_generator(lambda: keras_sequence_generator
                                        (df, FEATURE_COLS, scaler, SEQ_LEN, HORIZON_FRAMES,coordinate_targets, n_rows=N_ROWS, n_cols=N_COLS),
                                        output_types=output_types,output_shapes=output_shapes)

    if shuffle:
        ds = ds.shuffle(2048)
        # ds = ds.repeat()
    ds = ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

    return ds


## Velocity addition Func

In [10]:

def add_velocity_features(df):
    """Enhanced velocity and movement features for better coordinate prediction"""
    df = df.sort_values(["match_id", "player_id", "frame_number"])

    # === BASIC VELOCITY (existing) ===
    df["dx"] = df.groupby(["match_id", "player_id"])["x_normalized"].diff().fillna(0)
    df["dy"] = df.groupby(["match_id", "player_id"])["y_normalized"].diff().fillna(0)

    # Smooth velocities
    df["dx"] = df.groupby(["match_id", "player_id"])["dx"].rolling(3, min_periods=1).mean().reset_index(level=[0,1], drop=True)
    df["dy"] = df.groupby(["match_id", "player_id"])["dy"].rolling(3, min_periods=1).mean().reset_index(level=[0,1], drop=True)

    # === MULTI-WINDOW VELOCITY (NEW) ===
    for window in [3, 5, 10]:
        df[f"dx_avg_{window}"] = df.groupby(["match_id", "player_id"])["dx"].rolling(window, min_periods=1).mean().reset_index(level=[0,1], drop=True)
        df[f"dy_avg_{window}"] = df.groupby(["match_id", "player_id"])["dy"].rolling(window, min_periods=1).mean().reset_index(level=[0,1], drop=True)
        df[f"speed_avg_{window}"] = np.sqrt(df[f"dx_avg_{window}"]**2 + df[f"dy_avg_{window}"]**2)

    # === EXISTING FEATURES ===
    df["speed_normalized"] = np.sqrt(df["dx"]**2 + df["dy"]**2)
    df["acceleration"] = df.groupby(["match_id", "player_id"])["speed_normalized"].diff().fillna(0)
    df["movement_angle"] = np.arctan2(df["dy"], df["dx"])

    # === NEW: ACCELERATION TRENDS ===
    df["acceleration_trend"] = df.groupby(["match_id", "player_id"])["acceleration"].rolling(5, min_periods=1).mean().reset_index(level=[0,1], drop=True)
    
    # === NEW: DIRECTION PERSISTENCE ===
    df["angle_change"] = df.groupby(["match_id", "player_id"])["movement_angle"].diff().fillna(0)
    df["angle_stability"] = df.groupby(["match_id", "player_id"])["angle_change"].rolling(5, min_periods=1).std().reset_index(level=[0,1], drop=True).fillna(0)
    
    # === NEW: SPEED MOMENTUM ===
    df["speed_change_rate"] = df.groupby(["match_id", "player_id"])["speed_normalized"].diff().fillna(0)

    # === SPATIAL FEATURES (existing) ===
    df["distance_from_center"] = np.sqrt((df["x_normalized"] - 0.5)**2 + (df["y_normalized"] - 0.5)**2)
    df["distance_from_goal_home"] = df["x_normalized"]
    df["distance_from_goal_away"] = 1 - df["x_normalized"]
    df["distance_from_sideline"] = np.minimum(df["y_normalized"], 1 - df["y_normalized"])

    # === BALL & TEAM FEATURES (existing) ===
    if "team_spread" not in df.columns:
        df["team_spread"] = df.groupby(["match_id", "frame_number", "team_type"])["x_normalized"].transform("std").fillna(0)
    if "distance_to_ball" not in df.columns:
        df["distance_to_ball"] = 0.5
    if "ball_possession_proximity" not in df.columns:
        df["ball_possession_proximity"] = df["distance_to_ball"]

    print("\n📌 Enhanced velocity features added!")
    print(f"New multi-window features: dx_avg_3/5/10, dy_avg_3/5/10, speed_avg_3/5/10")
    print(f"New trend features: acceleration_trend, angle_change, angle_stability, speed_change_rate")
    
    return df

In [11]:


# def add_velocity_features(df):
#     """Add velocity and enhanced features for better zone prediction"""
#     df = df.sort_values(["match_id", "player_id", "frame_number"])

#     # Basic velocity features
#     df["dx"] = df.groupby(["match_id", "player_id"])["x_normalized"].diff().fillna(0)
#     df["dy"] = df.groupby(["match_id", "player_id"])["y_normalized"].diff().fillna(0)

#     # Smooth velocities to reduce noise
#     df["dx"] = df.groupby(["match_id", "player_id"])["dx"].rolling(3, min_periods=1).mean().reset_index(level=[0,1], drop=True)
#     df["dy"] = df.groupby(["match_id", "player_id"])["dy"].rolling(3, min_periods=1).mean().reset_index(level=[0,1], drop=True)

#     # Speed and acceleration
#     df["speed_normalized"] = np.sqrt(df["dx"]**2 + df["dy"]**2)
#     df["acceleration"] = df.groupby(["match_id", "player_id"])["speed_normalized"].diff().fillna(0)

#     # Direction of movement (angle in radians)
#     df["movement_angle"] = np.arctan2(df["dy"], df["dx"])

#     # Distance from center of field
#     df["distance_from_center"] = np.sqrt((df["x_normalized"] - 0.5)**2 + (df["y_normalized"] - 0.5)**2)

#     # Proximity to field boundaries
#     df["distance_from_goal_home"] = df["x_normalized"]  # distance from home goal (x=0)
#     df["distance_from_goal_away"] = 1 - df["x_normalized"]  # distance from away goal (x=1)
#     df["distance_from_sideline"] = np.minimum(df["y_normalized"], 1 - df["y_normalized"])

#     # Team-based features (if available)
#     if "team_has_possession" in df.columns:
#         # Time since possession change
#         df["possession_change"] = df.groupby(["match_id"])["team_has_possession"].diff().fillna(0)
#         df["frames_since_possession_change"] = df.groupby(["match_id"]).apply(
#             lambda x: (x["possession_change"] != 0).cumsum()
#         ).reset_index(level=0, drop=True)

#     # Player density in surrounding area (simplified)
#     if "team_spread" not in df.columns:
#         # Calculate a simple team spread measure
#         df["team_spread"] = df.groupby(["match_id", "frame_number", "team_type"])["x_normalized"].transform("std").fillna(0)

#     # Ball proximity features
#     if "distance_to_ball" not in df.columns:
#         df["distance_to_ball"] = 0.5  # placeholder if not available

#     if "ball_possession_proximity" not in df.columns:
#         df["ball_possession_proximity"] = df["distance_to_ball"]  # simplified version
#     print("\n📌 add_velocity_features sample:")
#     print(df.head(5)[["x_normalized", "y_normalized","dx", 
#                       "dy","speed_normalized", "acceleration","movement_angle"]])
#     return df

def add_contextual_features(df):
    """Add contextual features based on game state"""
    # Time-based features
    df["period_progress"] = df["timestamp_seconds"] / df.groupby(["match_id", "period"])["timestamp_seconds"].transform("max")

    # Formation-based features (simplified)
    if "position_category" in df.columns:
        # Create position-based features
        position_encoding = {"goalkeeper": 0, "defender": 1, "midfielder": 2, "forward": 3, "unknown": 4}
        df["position_encoded"] = df["position_category"].map(position_encoding).fillna(4)
    else:
        df["position_encoded"] = 2  # default to midfielder
    print("\n📌 add_contextual_features sample:")
    print(df.head(5)[[
        "period_progress",
        "position_encoded"
    ]])

    return df

## Original Locus Lab TCN

In [12]:


# 1. The Helper Block (Padding)
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

# 2. The Residual Block
class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

# 3. The Main TCN Backbone (Renamed to avoid conflict)
class LocusLabTCN(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super(LocusLabTCN, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1,
                                     dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size,
                                     dropout=dropout)]

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# 4. Your Final Model Wrapper
class DeltaTCN(nn.Module):
    def __init__(self, input_dim, num_classes=2, channels=[64, 128, 256, 512], kernel_size=3, dropout=0.3):
        super().__init__()
        # Initialize the specific TCN class defined above
        self.tcn = LocusLabTCN(input_dim, channels, kernel_size=kernel_size, dropout=dropout)
        self.co_coords = CO_ORDINATES

        # Output heads
        if self.co_coords:
            self.delta_head = nn.Linear(channels[-1], num_classes)          # dx, dy
        else:
            self.zone_head = nn.Linear(channels[-1], NUM_ZONES)   # classification over 144 zones

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        x = x.transpose(1, 2)  # (batch, input_dim, seq_len)
        out = self.tcn(x)
        out_last = out[:, :, -1]  # take last timestep for prediction

        if self.co_coords:
            return self.delta_head(out_last)      # regression: dx, dy
        else:
            return self.zone_head(out_last)       # classification: zones

## Keras TCN enhanced

In [13]:

# @misc{KerasTCN,
#   author = {Philippe Remy},
#   title = {Temporal Convolutional Networks for Keras},
#   year = {2020},
#   publisher = {GitHub},
#   journal = {GitHub repository},
#   howpublished = {\url{https://github.com/philipperemy/keras-tcn}},
# }

try:
    # pylint: disable=E0611,E0401
    from keras.src.saving import register_keras_serializable  # For recent Keras
except ImportError:
    # pylint: disable=E0611,E0401
    from tensorflow.keras.saving import register_keras_serializable  # For older versions

# pylint: disable=E0611,E0401
from tensorflow.keras import backend as K, Model, Input, optimizers
# pylint: disable=E0611,E0401
from tensorflow.keras import layers
# pylint: disable=E0611,E0401
from tensorflow.keras.layers import Activation, SpatialDropout1D, Lambda
# pylint: disable=E0611,E0401
from tensorflow.keras.layers import Layer, Conv1D, Dense, BatchNormalization, LayerNormalization


def is_power_of_two(num: int):
    return num != 0 and ((num & (num - 1)) == 0)


def adjust_dilations(dilations: list):
    if all([is_power_of_two(i) for i in dilations]):
        return dilations
    else:
        new_dilations = [2 ** i for i in dilations]
        return new_dilations


class ResidualBlock(Layer):

    def __init__(self,
                 dilation_rate: int,
                 nb_filters: int,
                 kernel_size: int,
                 padding: str,
                 activation: str = 'relu',
                 dropout_rate: float = 0,
                 kernel_initializer: str = 'he_normal',
                 use_batch_norm: bool = False,
                 use_layer_norm: bool = False,
                 **kwargs):
        """Defines the residual block for the WaveNet TCN
        Args:
            x: The previous layer in the model
            training: boolean indicating whether the layer should behave in training mode or in inference mode
            dilation_rate: The dilation power of 2 we are using for this residual block
            nb_filters: The number of convolutional filters to use in this block
            kernel_size: The size of the convolutional kernel
            padding: The padding used in the convolutional layers, 'same' or 'causal'.
            activation: The final activation used in o = Activation(x + F(x))
            dropout_rate: Float between 0 and 1. Fraction of the input units to drop.
            kernel_initializer: Initializer for the kernel weights matrix (Conv1D).
            use_batch_norm: Whether to use batch normalization in the residual layers or not.
            use_layer_norm: Whether to use layer normalization in the residual layers or not.
            kwargs: Any initializers for Layer class.
        """

        self.dilation_rate = dilation_rate
        self.nb_filters = nb_filters
        self.kernel_size = kernel_size
        self.padding = padding
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.use_batch_norm = use_batch_norm
        self.use_layer_norm = use_layer_norm
        self.kernel_initializer = kernel_initializer
        self.layers = []
        self.shape_match_conv = None
        self.res_output_shape = None
        self.final_activation = None
        self.batch_norm_layers = []
        self.layer_norm_layers = []

        super(ResidualBlock, self).__init__(**kwargs)

    def _build_layer(self, layer):
        """Helper function for building layer
        Args:
            layer: Appends layer to internal layer list and builds it based on the current output
                   shape of ResidualBlocK. Updates current output shape.
        """
        self.layers.append(layer)
        self.layers[-1].build(self.res_output_shape)
        self.res_output_shape = self.layers[-1].compute_output_shape(self.res_output_shape)

    def build(self, input_shape):

        with K.name_scope(self.name):  # name scope used to make sure weights get unique names
            self.layers = []
            self.res_output_shape = input_shape

            for k in range(2):  # dilated conv block.
                name = 'conv1D_{}'.format(k)
                with K.name_scope(name):  # name scope used to make sure weights get unique names
                    conv = Conv1D(
                        filters=self.nb_filters,
                        kernel_size=self.kernel_size,
                        dilation_rate=self.dilation_rate,
                        padding=self.padding,
                        name=name,
                        kernel_initializer=self.kernel_initializer
                    )
                    self._build_layer(conv)

                with K.name_scope('norm_{}'.format(k)):
                    if self.use_batch_norm:
                        bn_layer = BatchNormalization()
                        self.batch_norm_layers.append(bn_layer)
                        self._build_layer(bn_layer)
                    elif self.use_layer_norm:
                        ln_layer = LayerNormalization()
                        self.layer_norm_layers.append(ln_layer)
                        self._build_layer(ln_layer)

                with K.name_scope('act_and_dropout_{}'.format(k)):
                    self._build_layer(Activation(self.activation, name='Act_Conv1D_{}'.format(k)))
                    self._build_layer(SpatialDropout1D(rate=self.dropout_rate, name='SDropout_{}'.format(k)))

            if self.nb_filters != input_shape[-1]:
                # 1x1 conv to match the shapes (channel dimension).
                name = 'matching_conv1D'
                with K.name_scope(name):
                    # make and build this layer separately because it directly uses input_shape.
                    # 1x1 conv.
                    self.shape_match_conv = Conv1D(
                        filters=self.nb_filters,
                        kernel_size=1,
                        padding='same',
                        name=name,
                        kernel_initializer=self.kernel_initializer
                    )
            else:
                name = 'matching_identity'
                self.shape_match_conv = Lambda(lambda x: x, name=name)

            with K.name_scope(name):
                self.shape_match_conv.build(input_shape)
                self.res_output_shape = self.shape_match_conv.compute_output_shape(input_shape)

            self._build_layer(Activation(self.activation, name='Act_Conv_Blocks'))
            self.final_activation = Activation(self.activation, name='Act_Res_Block')
            self.final_activation.build(self.res_output_shape)  # probably isn't necessary

            # this is done to force Keras to add the layers in the list to self._layers
            for layer in self.layers:
                self.__setattr__(layer.name, layer)
            self.__setattr__(self.shape_match_conv.name, self.shape_match_conv)
            self.__setattr__(self.final_activation.name, self.final_activation)

            super(ResidualBlock, self).build(input_shape)  # done to make sure self.built is set True

    def call(self, inputs, training=None, **kwargs):
        """
        Returns: A tuple where the first element is the residual model tensor, and the second
                 is the skip connection tensor.
        """
        # https://arxiv.org/pdf/1803.01271.pdf  page 4, Figure 1 (b).
        # x1: Dilated Conv -> Norm -> Dropout (x2).
        # x2: Residual (1x1 matching conv - optional).
        # Output: x1 + x2.
        # x1 -> connected to skip connections.
        # x1 + x2 -> connected to the next block.
        #       input
        #     x1      x2
        #   conv1D    1x1 Conv1D (optional)
        #    ...
        #   conv1D
        #    ...
        #       x1 + x2
        x1 = inputs
        for layer in self.layers:
            training_flag = 'training' in dict(inspect.signature(layer.call).parameters)
            x1 = layer(x1, training=training) if training_flag else layer(x1)
        x2 = self.shape_match_conv(inputs)
        x1_x2 = self.final_activation(layers.add([x2, x1], name='Add_Res'))
        return [x1_x2, x1]

    def compute_output_shape(self, input_shape):
        return [self.res_output_shape, self.res_output_shape]


@register_keras_serializable()
class TCN(Layer):
    """Creates a TCN layer.

        Input shape:
            A 3D tensor with shape (batch_size, timesteps, input_dim).

        Args:
            nb_filters: The number of filters to use in the convolutional layers. Can be a list.
            kernel_size: The size of the kernel to use in each convolutional layer.
            dilations: The list of the dilations. Example is: [1, 2, 4, 8, 16, 32, 64].
            nb_stacks : The number of stacks of residual blocks to use.
            padding: The padding to use in the convolutional layers, 'causal' or 'same'.
            use_skip_connections: Boolean. If we want to add skip connections from input to each residual blocK.
            return_sequences: Boolean. Whether to return the last output in the output sequence, or the full sequence.
            activation: The activation used in the residual blocks o = Activation(x + F(x)).
            dropout_rate: Float between 0 and 1. Fraction of the input units to drop.
            kernel_initializer: Initializer for the kernel weights matrix (Conv1D).
            use_batch_norm: Whether to use batch normalization in the residual layers or not.
            use_layer_norm: Whether to use layer normalization in the residual layers or not.
            go_backwards: Boolean (default False). If True, process the input sequence backwards and
            return the reversed sequence.
            return_state: Boolean. Whether to return the last state in addition to the output. Default: False.
            kwargs: Any other arguments for configuring parent class Layer. For example "name=str", Name of the model.
                    Use unique names when using multiple TCN.
        Returns:
            A TCN layer.
        """

    def __init__(self,
                 nb_filters=64,
                 kernel_size=3,
                 nb_stacks=1,
                 dilations=(1, 2, 4, 8, 16, 32),
                 padding='causal',
                 use_skip_connections=True,
                 dropout_rate=0.0,
                 return_sequences=False,
                 activation='relu',
                 kernel_initializer='he_normal',
                 use_batch_norm=False,
                 use_layer_norm=False,
                 go_backwards=False,
                 return_state=False,
                 **kwargs):
        self.stateful = False  # TCN are not stateful. Keras requires this parameter.
        self.return_sequences = return_sequences
        self.dropout_rate = dropout_rate
        self.use_skip_connections = use_skip_connections
        self.dilations = dilations
        self.nb_stacks = nb_stacks
        self.kernel_size = kernel_size
        self.nb_filters = nb_filters
        self.activation_name = activation
        self.padding = padding
        self.kernel_initializer = kernel_initializer
        self.use_batch_norm = use_batch_norm
        self.use_layer_norm = use_layer_norm
        self.go_backwards = go_backwards
        self.return_state = return_state
        self.skip_connections = []
        self.residual_blocks = []
        self.layers_outputs = []
        self.build_output_shape = None
        self.slicer_layer = None  # in case return_sequence=False
        self.output_slice_index = None  # in case return_sequence=False
        self.padding_same_and_time_dim_unknown = False  # edge case if padding='same' and time_dim = None

        if self.use_batch_norm + self.use_layer_norm > 1:
            raise ValueError('Only one normalization can be specified at once.')

        if isinstance(self.nb_filters, list):
            assert len(self.nb_filters) == len(self.dilations)
            if len(set(self.nb_filters)) > 1 and self.use_skip_connections:
                raise ValueError('Skip connections are not compatible '
                                 'with a list of filters, unless they are all equal.')

        if padding != 'causal' and padding != 'same':
            raise ValueError("Only 'causal' or 'same' padding are compatible for this layer.")

        # initialize parent class
        super(TCN, self).__init__(**kwargs)

    @property
    def receptive_field(self):
        return 1 + 2 * (self.kernel_size - 1) * self.nb_stacks * sum(self.dilations)

    def tolist(self, shape):
        # noinspection PyBroadException
        try:
            return shape.as_list()
        except Exception:
            return list(shape)

    def build(self, input_shape):

        # member to hold current output shape of the layer for building purposes
        self.build_output_shape = input_shape

        # list to hold all the member ResidualBlocks
        self.residual_blocks = []
        total_num_blocks = self.nb_stacks * len(self.dilations)
        if not self.use_skip_connections:
            total_num_blocks += 1  # cheap way to do a false case for below

        for s in range(self.nb_stacks):
            for i, d in enumerate(self.dilations):
                res_block_filters = self.nb_filters[i] if isinstance(self.nb_filters, list) else self.nb_filters
                self.residual_blocks.append(ResidualBlock(dilation_rate=d,
                                                          nb_filters=res_block_filters,
                                                          kernel_size=self.kernel_size,
                                                          padding=self.padding,
                                                          activation=self.activation_name,
                                                          dropout_rate=self.dropout_rate,
                                                          use_batch_norm=self.use_batch_norm,
                                                          use_layer_norm=self.use_layer_norm,
                                                          kernel_initializer=self.kernel_initializer,
                                                          name='residual_block_{}'.format(len(self.residual_blocks))))
                # build newest residual block
                self.residual_blocks[-1].build(self.build_output_shape)
                self.build_output_shape = self.residual_blocks[-1].res_output_shape

        # this is done to force keras to add the layers in the list to self._layers
        for layer in self.residual_blocks:
            self.__setattr__(layer.name, layer)

        self.output_slice_index = None
        if self.padding == 'same':
            time = self.tolist(self.build_output_shape)[1]
            if time is not None:  # if time dimension is defined. e.g. shape = (bs, 500, input_dim).
                self.output_slice_index = int(self.tolist(self.build_output_shape)[1] / 2)
            else:
                # It will known at call time. c.f. self.call.
                self.padding_same_and_time_dim_unknown = True

        else:
            self.output_slice_index = -1  # causal case.
        self.slicer_layer = Lambda(lambda tt: tt[:, self.output_slice_index, :], name='Slice_Output')
        self.slicer_layer.build(self.tolist(self.build_output_shape))

    def compute_output_shape(self, input_shape):
        """
        Overridden in case keras uses it somewhere... no idea. Just trying to avoid future errors.
        """
        if not self.built:
            self.build(input_shape)
        if not self.return_sequences:
            batch_size = self.build_output_shape[0]
            batch_size = batch_size.value if hasattr(batch_size, 'value') else batch_size
            nb_filters = self.build_output_shape[-1]
            return [batch_size, nb_filters]
        else:
            # Compatibility tensorflow 1.x
            return [v.value if hasattr(v, 'value') else v for v in self.build_output_shape]

    def call(self, inputs, training=None, **kwargs):
        x = inputs

        if self.go_backwards:
            # reverse x in the time axis
            x = tf.reverse(x, axis=[1])

        self.layers_outputs = [x]
        self.skip_connections = []
        for res_block in self.residual_blocks:
            try:
                x, skip_out = res_block(x, training=training)
            except TypeError:  # compatibility with tensorflow 1.x
                x, skip_out = res_block(K.cast(x, 'float32'), training=training)
            self.skip_connections.append(skip_out)
            self.layers_outputs.append(x)

        if self.use_skip_connections:
            if len(self.skip_connections) > 1:
                # Keras: A merge layer should be called on a list of at least 2 inputs. Got 1 input.
                x = layers.add(self.skip_connections, name='Add_Skip_Connections')
            else:
                x = self.skip_connections[0]
            self.layers_outputs.append(x)

        if not self.return_sequences:
            # case: time dimension is unknown. e.g. (bs, None, input_dim).
            if self.padding_same_and_time_dim_unknown:
                self.output_slice_index = K.shape(self.layers_outputs[-1])[1] // 2
            x = self.slicer_layer(x)
            self.layers_outputs.append(x)
        return x

    def get_config(self):
        """
        Returns the config of a the layer. This is used for saving and loading from a model
        :return: python dictionary with specs to rebuild layer
        """
        config = super(TCN, self).get_config()
        config['nb_filters'] = self.nb_filters
        config['kernel_size'] = self.kernel_size
        config['nb_stacks'] = self.nb_stacks
        config['dilations'] = self.dilations
        config['padding'] = self.padding
        config['use_skip_connections'] = self.use_skip_connections
        config['dropout_rate'] = self.dropout_rate
        config['return_sequences'] = self.return_sequences
        config['activation'] = self.activation_name
        config['use_batch_norm'] = self.use_batch_norm
        config['use_layer_norm'] = self.use_layer_norm
        config['kernel_initializer'] = self.kernel_initializer
        config['go_backwards'] = self.go_backwards
        config['return_state'] = self.return_state
        return config


def compiled_tcn(num_feat,  # type: int
                 num_classes,  # type: int
                 nb_filters,  # type: int
                 kernel_size,  # type: int
                 dilations,  # type: List[int]
                 nb_stacks,  # type: int
                 max_len,  # type: int
                 output_len=1,  # type: int
                 padding='causal',  # type: str
                 use_skip_connections=False,  # type: bool
                 return_sequences=True,
                 regression=False,  # type: bool
                 dropout_rate=0.05,  # type: float
                 name='tcn',  # type: str,
                 kernel_initializer='he_normal',  # type: str,
                 activation='relu',  # type:str,
                 opt='adam',
                 lr=0.002,
                 use_batch_norm=False,
                 use_layer_norm=False):
    # type: (...) -> Model
    """Creates a compiled TCN model for a given task (i.e. regression or classification).
    Classification uses a sparse categorical loss. Please input class ids and not one-hot encodings.

    Args:
        num_feat: The number of features of your input, i.e. the last dimension of: (batch_size, timesteps, input_dim).
        num_classes: The size of the final dense layer, how many classes we are predicting.
        nb_filters: The number of filters to use in the convolutional layers.
        kernel_size: The size of the kernel to use in each convolutional layer.
        dilations: The list of the dilations. Example is: [1, 2, 4, 8, 16, 32, 64].
        nb_stacks : The number of stacks of residual blocks to use.
        max_len: The maximum sequence length, use None if the sequence length is dynamic.
        padding: The padding to use in the convolutional layers.
        use_skip_connections: Boolean. If we want to add skip connections from input to each residual blocK.
        return_sequences: Boolean. Whether to return the last output in the output sequence, or the full sequence.
        regression: Whether the output should be continuous or discrete.
        dropout_rate: Float between 0 and 1. Fraction of the input units to drop.
        activation: The activation used in the residual blocks o = Activation(x + F(x)).
        name: Name of the model. Useful when having multiple TCN.
        kernel_initializer: Initializer for the kernel weights matrix (Conv1D).
        opt: Optimizer name.
        lr: Learning rate.
        use_batch_norm: Whether to use batch normalization in the residual layers or not.
        use_layer_norm: Whether to use layer normalization in the residual layers or not.
    Returns:
        A compiled keras TCN.
    """

    dilations = adjust_dilations(dilations)

    input_layer = Input(shape=(max_len, num_feat))

    x = TCN(nb_filters, kernel_size, nb_stacks, dilations, padding,
            use_skip_connections, dropout_rate, return_sequences,
            activation, kernel_initializer, use_batch_norm, use_layer_norm,
            name=name)(input_layer)

    print('x.shape=', x.shape)

    def get_opt():
        if opt == 'adam':
            return optimizers.Adam(learning_rate=lr, clipnorm=1.)
        elif opt == 'rmsprop':
            return optimizers.RMSprop(learning_rate=lr, clipnorm=1.)
        else:
            raise Exception('Only Adam and RMSProp are available here')

    if not regression:
        # classification
        x = Dense(num_classes)(x)
        x = Activation('softmax')(x)
        output_layer = x
        model = Model(input_layer, output_layer)

        # https://github.com/keras-team/keras/pull/11373
        # It's now in Keras@master but still not available with pip.
        # TODO remove later.
        def accuracy(y_true, y_pred):
            # reshape in case it's in shape (num_samples, 1) instead of (num_samples,)
            if K.ndim(y_true) == K.ndim(y_pred):
                y_true = K.squeeze(y_true, -1)
            # convert dense predictions to labels
            y_pred_labels = K.argmax(y_pred, axis=-1)
            y_pred_labels = K.cast(y_pred_labels, K.floatx())
            return K.cast(K.equal(y_true, y_pred_labels), K.floatx())

        model.compile(get_opt(), loss='sparse_categorical_crossentropy', metrics=[accuracy])
    else:
        # regression
        x = Dense(num_classes)(x) 
        x = Activation('linear')(x)

        output_layer = x
        model = Model(input_layer, output_layer)
        model.compile(get_opt(), loss='mean_squared_error', metrics=['mae'])
    print('model.x = {}'.format(input_layer.shape))
    print('model.y = {}'.format(output_layer.shape))
    return model


def tcn_full_summary(model: Model, expand_residual_blocks=True):
    import tensorflow as tf
    # 2.6.0-rc1, 2.5.0...
    versions = [int(v) for v in tf.__version__.split('-')[0].split('.')]
    if versions[0] <= 2 and versions[1] < 5:
        layers = model._layers.copy()  # store existing layers
        model._layers.clear()  # clear layers

        for i in range(len(layers)):
            if isinstance(layers[i], TCN):
                for layer in layers[i]._layers:
                    if not isinstance(layer, ResidualBlock):
                        if not hasattr(layer, '__iter__'):
                            model._layers.append(layer)
                    else:
                        if expand_residual_blocks:
                            for lyr in layer._layers:
                                if not hasattr(lyr, '__iter__'):
                                    model._layers.append(lyr)
                        else:
                            model._layers.append(layer)
            else:
                model._layers.append(layers[i])

        model.summary()  # print summary

        # restore original layers
        model._layers.clear()
        [model._layers.append(lyr) for lyr in layers]
    else:
        print('WARNING: tcn_full_summary: Compatible with tensorflow 2.5.0 or below.')
        print('Use tensorboard instead. Example in keras-tcn/tasks/tcn_tensorboard.py.')

In [14]:
#CUSTOM LOSS FUNCTIONS FOR BETTER COORDINATE PREDICTION

def huber_loss_tf(delta=0.1):
    """
    Huber loss - less sensitive to outliers than MSE
    Better for coordinate prediction with occasional large errors
    """
    def loss(y_true, y_pred):
        error = y_true - y_pred
        is_small = tf.abs(error) <= delta
        squared_loss = 0.5 * tf.square(error)
        linear_loss = delta * (tf.abs(error) - 0.5 * delta)
        return tf.where(is_small, squared_loss, linear_loss)
    return loss

def weighted_mse_loss():
    """
    Weighted MSE that penalizes x and y errors differently
    (x-axis errors might be more critical in football)
    """
    def loss(y_true, y_pred):
        x_error = tf.square(y_true[:, 0] - y_pred[:, 0])
        y_error = tf.square(y_true[:, 1] - y_pred[:, 1])
        return 1.2 * x_error + 0.8 * y_error  # weight x-direction more
    return loss

print("✓ Custom loss functions loaded")

✓ Custom loss functions loaded


##     Enhanced model combining LSTM for temporal patterns and CNN for feature extraction


In [15]:
class EnhancedFootballModel(nn.Module):
    """
    Enhanced model combining LSTM for temporal patterns and CNN for feature extraction
    """
    def __init__(self, input_dim, num_classes, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()

        # Feature extraction layer
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # LSTM for temporal modeling
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # Attention mechanism to focus on important time steps
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim * 2,  # bidirectional
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )

        # Final classification layers
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LSTM):
                for name, param in module.named_parameters():
                    if 'weight' in name:
                        nn.init.xavier_uniform_(param)
                    elif 'bias' in name:
                        nn.init.zeros_(param)

    # def forward(self, x):
    #     # x shape: (batch_size, seq_len, input_dim)
    #     batch_size, seq_len, input_dim = x.shape

    #     # Extract features for each time step
    #     x_reshaped = x.view(-1, input_dim)  # (batch_size * seq_len, input_dim)
    #     features = self.feature_extractor(x_reshaped)  # (batch_size * seq_len, hidden_dim)
    #     features = features.view(batch_size, seq_len, -1)  # (batch_size, seq_len, hidden_dim)

    #     # LSTM processing
    #     lstm_out, (hidden, cell) = self.lstm(features)  # (batch_size, seq_len, hidden_dim * 2)

    #     # Attention mechanism
    #     attended_out, attention_weights = self.attention(lstm_out, lstm_out, lstm_out)

    #     # Use the last time step for classification
    #     final_features = attended_out[:, -1, :]  # (batch_size, hidden_dim * 2)

    #     # Classification
    #     logits = self.classifier(final_features)  # (batch_size, num_classes)

    #     return logits

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        batch_size, seq_len, input_dim = x.shape

        # 1. Feature extraction for each time step
        x_reshaped = x.view(-1, input_dim)
        features = self.feature_extractor(x_reshaped)
        features = features.view(batch_size, seq_len, -1)

        # 2. LSTM processing
        lstm_out, _ = self.lstm(features) # (batch_size, seq_len, hidden_dim * 2)

        # 3. Attention mechanism
        # attended_out shape: (batch_size, seq_len, hidden_dim * 2)
        attended_out, _ = self.attention(lstm_out, lstm_out, lstm_out)

        # 4. Global Average Pooling (The upgrade)
        # We take the mean across the sequence (dim 1)
        # This captures the "essence" of the whole sequence after attention weights it
        avg_pool = torch.mean(attended_out, dim=1) 
        max_pool, _ = torch.max(attended_out, dim=1)
        
        # Combining Avg and Max pooling is a common trick to get the best of both worlds
        final_features = avg_pool + max_pool 

        # 5. Classification
        logits = self.classifier(final_features)

        return logits


## Improved Temporal Convolutional Network (not verified)

In [16]:
 class TemporalConvNet(nn.Module):
    """Improved Temporal Convolutional Network"""
    def __init__(self, input_dim, num_classes, num_channels=[64, 128, 256], dropout=0.3):
        super().__init__()
        layers = []
        in_ch = input_dim

        for i, out_ch in enumerate(num_channels):
            dilation = 2 ** i  # increasing dilation
            layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=dilation, dilation=dilation),
                nn.BatchNorm1d(out_ch),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            in_ch = out_ch

        self.network = nn.Sequential(*layers)

        # Global pooling and classification
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(in_ch, in_ch // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(in_ch // 2, num_classes)
        )

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        x = x.transpose(1, 2)  # (batch_size, input_dim, seq_len)
        x = self.network(x)
        x = self.global_pool(x).squeeze(-1)  # (batch_size, channels)
        return self.classifier(x)


##  Unified training function for PyTorch or Keras backend.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, patience=PATIENCE):
   
    if BACKEND == "pytorch":
     
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if torch.cuda.device_count() > 1:
            model = torch.nn.DataParallel(model)
        model.to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = torch.nn.MSELoss() if CO_ORDINATES else torch.nn.CrossEntropyLoss()
         # LR scheduler for classification
        scheduler = None
        if not CO_ORDINATES:
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=patience // 2
            )
        best_val_metric = float('inf') if CO_ORDINATES else 0.0
        patience_counter = 0
        history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_top3_acc': []}
        
        print(f"Starting training on {device} for {epochs} epochs...")

        def get_targets(yb):
            if isinstance(yb, dict):
                yb = {k: v.to(device) for k, v in yb.items()}
                return yb["delta"] if CO_ORDINATES else yb["zone"]
            else:
                return yb.to(device)

        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            correct, total = 0, 0

            for Xb, yb in train_loader:
                Xb = Xb.to(device)
                targets = get_targets(yb)

                optimizer.zero_grad()
                outputs = model(Xb)
                loss = criterion(outputs, targets)
                loss.backward()
                
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                running_loss += loss.item() * Xb.size(0)
                
                if not CO_ORDINATES:
                    _, preds = outputs.max(1)
                    correct += (preds == targets).sum().item()
                    total += targets.size(0)
                    
                if (batch_idx + 1) % 100 == 0:
                    if not CO_ORDINATES:
                        batch_acc = correct / max(1, total)
                        print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch_idx+1}/{len(train_loader)}] "
                          f"Loss: {loss.item():.4f} Acc: {batch_acc:.4f}")
                    else:
                        print(f"Epoch [{epoch+1}/{epochs}] Batch [{batch_idx+1}/{len(train_loader)}] Loss: {loss.item():.4f}")

            train_loss = running_loss / len(train_loader.dataset)
            train_acc = (correct / total) if not CO_ORDINATES else None

            # Validation
            model.eval()
            val_loss = 0.0
            val_correct, val_total = 0, 0
            all_probs = []

            with torch.no_grad():
                for Xb, yb in val_loader:
                    Xb = Xb.to(device)
                    targets = get_targets(yb)
                    
                    outputs = model(Xb)
                    loss = criterion(outputs, targets)
                    val_loss += loss.item() * Xb.size(0)
                    
                    if not CO_ORDINATES:
                        probs = F.softmax(outputs, dim=1)
                        all_probs.append(probs.cpu())
                        _, preds = outputs.max(1)
                        val_correct += (preds == targets).sum().item()
                        val_total += targets.size(0)

            val_loss /= len(val_loader.dataset)
            val_acc = (val_correct / val_total) if not CO_ORDINATES else None

            
            val_top3 = None
            if not CO_ORDINATES and all_probs:
                val_top3 = top_k_accuracy_score(
                    torch.cat(all_labels).numpy(), 
                    torch.cat(all_probs).numpy(), 
                    k=3
                )

            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_acc'].append(val_acc)
            history['val_top3_acc'].append(val_top3)
            
            # Early stopping
            current_metric = val_loss if CO_ORDINATES else val_acc
            improved = (current_metric < best_val_metric) if CO_ORDINATES else (current_metric > best_val_metric)
            if improved:
                best_val_metric = current_metric
                best_model_wts = copy.deepcopy(model.state_dict())
                patience_counter = 0
                print(f"New Best: {best_val_metric:.4f}")
            else:
                patience_counter += 1
                
            if not CO_ORDINATES and scheduler:
                scheduler.step(val_acc)
                 
            if patience_counter >= patience:
                break

        model.load_state_dict(best_model_wts)
        return model, history

    elif BACKEND == "keras":

        # Determine loss
        if CO_ORDINATES:
            loss = "mse"
            metrics = ["mae"]
        else:
            loss = "sparse_categorical_crossentropy"
            metrics = ["accuracy"]

        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
        model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

        callbacks = [
            tf.keras.callbacks.EarlyStopping( monitor='val_mae' if CO_ORDINATES else 'val_accuracy',patience=5,restore_best_weights=True,mode='min' if CO_ORDINATES else 'max'),
            tf.keras.callbacks.ReduceLROnPlateau( monitor='val_mae' if CO_ORDINATES else 'val_accuracy',factor=0.5, patience=3,min_lr=1e-6, mode='min' if CO_ORDINATES else 'max'),
            tf.keras.callbacks.ModelCheckpoint('best_model_mae.keras', monitor='val_mae' if CO_ORDINATES else 'val_accuracy',save_best_only=True,mode='min' if CO_ORDINATES else 'max')
        ]

        history = model.fit(
            train_loader,
            validation_data=val_loader,
            epochs=epochs,
            callbacks=callbacks,
            verbose=1
        )

        return model, history


## Enhanced evaluation with multiple metrics      

In [18]:
def evaluate_with_metrics(model, loader):
   
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb)
            probs = F.softmax(logits, dim=1)

            _, preds = logits.max(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())
            all_probs.append(probs.cpu())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = torch.cat(all_probs, dim=0)

    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds)
    top3_acc = top_k_accuracy_score(all_labels, all_probs.numpy(), k=3)

    return acc, top3_acc, all_probs


def evaluate_detailed(model, loader, zone_names=None):
    """Detailed evaluation with classification report"""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb)
            _, preds = logits.max(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Accuracy metrics
    acc = accuracy_score(all_labels, all_preds)

    print(f"Overall Accuracy: {acc:.4f}")
    print("\nDetailed Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=zone_names))

    return acc, all_preds, all_labels


## Unified evaluation function for both regression (dx/dy) and classification (zones).

        
    Returns:
        - For regression: dict with 'mse', 'mae', 'rmse'
        - For classification: dict with 'accuracy', 'top3_accuracy', 'all_probs' 

In [ ]:
def evaluate_model(model, loader, co_ordinates=CO_ORDINATES, zone_names=None):

    model.eval()
    device = next(model.parameters()).device  # get model device automatically

    if co_ordinates:
        # Regression
        all_preds, all_targets = [], []
        criterion_mse = nn.MSELoss(reduction='sum')
        criterion_mae = nn.L1Loss(reduction='sum')

        with torch.no_grad():
            for Xb, yb in loader:
                Xb = Xb.to(device)
                yb = {k: v.to(device) for k, v in yb.items()}  # regression dict
                targets = yb["delta"]

                outputs = model(Xb)
                all_preds.append(outputs.cpu())
                all_targets.append(targets.cpu())

        all_preds = torch.cat(all_preds, dim=0)
        all_targets = torch.cat(all_targets, dim=0)

        mse = criterion_mse(all_preds, all_targets).item() / len(loader.dataset)
        mae = criterion_mae(all_preds, all_targets).item() / len(loader.dataset)
        rmse = mse ** 0.5

        metrics = {'mse': mse, 'mae': mae, 'rmse': rmse}

    else:
        # Classification
        all_preds, all_labels, all_probs = [], [], []

        with torch.no_grad():
            for Xb, yb in loader:
                Xb, yb = Xb.to(device), yb.to(device)
                outputs = model(Xb)
                if epoch == 0 and batch_idx == 0:
                    print("Outputs shape:", outputs.shape)
                    print("Target min:", targets_tensor.min().item(),"Target max:", targets_tensor.max().item())

                probs = F.softmax(outputs, dim=1)
                _, preds = outputs.max(1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(yb.cpu().numpy())
                all_probs.append(probs.cpu())

        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        all_probs = torch.cat(all_probs, dim=0)

        acc = accuracy_score(all_labels, all_preds)
        top3_acc = top_k_accuracy_score(all_labels, all_probs.numpy(), k=3)

        metrics = {'accuracy': acc, 'top3_accuracy': top3_acc, 'all_probs': all_probs}

        # Optional: print classification report
        if zone_names is not None:
            print("\nClassification Report:")
            print(classification_report(all_labels, all_preds, target_names=zone_names))

    return metrics



## CHECK FOR PREPROCESSED SEQUENCES AND LOAD IF AVAILABLE



In [20]:
def check_preprocessed_exists(backend='keras'):
    """Check if all required preprocessed files exist for the specified backend"""
    
    required_files = [
        f'{PREPROCESSED_DIR}/scaler.pkl',
        f'{PREPROCESSED_DIR}/config.pkl',
    ]
    
    # Add backend-specific files
    if backend == 'keras':
        required_files.extend([
            f'{PREPROCESSED_DIR}/sequences.h5',
            f'{PREPROCESSED_DIR}/train_df.pkl',
            f'{PREPROCESSED_DIR}/val_df.pkl',
            f'{PREPROCESSED_DIR}/test_df.pkl'
        ])
    else:  # pytorch
        required_files.extend([
            f'{PREPROCESSED_DIR}/train_dataset.pt',
            f'{PREPROCESSED_DIR}/val_dataset.pt',
            f'{PREPROCESSED_DIR}/test_dataset.pt'
        ])
    
    if not os.path.exists(PREPROCESSED_DIR):
        return False, f"Preprocessed directory not found: {PREPROCESSED_DIR}"
    
    missing_files = [f for f in required_files if not os.path.exists(f)]
    
    if missing_files:
        return False, f"Missing files for {backend}: {[os.path.basename(f) for f in missing_files]}"
    
    return True, f"All {backend} preprocessed files found"


def load_preprocessed_sequences_keras(input_dir=PREPROCESSED_DIR):
    """Load preprocessed sequences for Keras backend"""
    
    print("\n" + "="*80)
    print("📂 LOADING KERAS PREPROCESSED DATA")
    print("="*80)
    
    try:
        # 1. Load config first
        config = joblib.load(f'{input_dir}/config.pkl')
        print(f"✓ Loaded config")
        
        # 2. Load scaler
        scaler = joblib.load(f'{input_dir}/scaler.pkl')
        print(f"✓ Loaded scaler ({scaler.n_features_in_} features)")
        
        # 3. Load sequences
        print("Loading sequences from HDF5...")
        with h5py.File(f'{input_dir}/sequences.h5', 'r') as f:
            X_train = f['X_train'][:]
            y_train = f['y_train'][:]
            X_val = f['X_val'][:]
            y_val = f['y_val'][:]
            X_test = f['X_test'][:]
            y_test = f['y_test'][:]
        
        print(f"✓ Loaded sequences:")
        print(f"   Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
        
        # 4. Load dataframes
        train_df = pd.read_pickle(f'{input_dir}/train_df.pkl')
        val_df = pd.read_pickle(f'{input_dir}/val_df.pkl')
        test_df = pd.read_pickle(f'{input_dir}/test_df.pkl')
        
        print(f"✓ Loaded dataframes:")
        print(f"   Train: {len(train_df)} rows, Val: {len(val_df)} rows, Test: {len(test_df)} rows")
        
        # 5. Update global variables
        global SEQ_LEN, HORIZON_FRAMES, HORIZON_SECONDS, N_ROWS, N_COLS
        global NUM_ZONES, FEATURE_COLS, CO_ORDINATES, BATCH_SIZE, BACKEND
        
        SEQ_LEN = config['seq_len']
        HORIZON_FRAMES = config['horizon_frames']
        HORIZON_SECONDS = config['horizon_seconds']
        N_ROWS = config['n_rows']
        N_COLS = config['n_cols']
        NUM_ZONES = N_ROWS * N_COLS
        FEATURE_COLS = config['feature_cols']
        CO_ORDINATES = config['coordinate_targets']
        BATCH_SIZE = config['batch_size']
        BACKEND = config.get('backend', 'keras')
        
        print(f"\n✓ Configuration updated:")
        print(f"   SEQ_LEN={SEQ_LEN}, HORIZON={HORIZON_FRAMES} frames ({HORIZON_SECONDS}s)")
        print(f"   Grid: {N_ROWS}x{N_COLS}={NUM_ZONES} zones")
        print(f"   Features: {len(FEATURE_COLS)}")
        print(f"   Coordinate targets: {CO_ORDINATES}")
        
        print("\n" + "="*80)
        print("✅ KERAS PREPROCESSED DATA LOADED SUCCESSFULLY")
        print("="*80)
        
        return {
            'X_train': X_train, 'y_train': y_train,
            'X_val': X_val, 'y_val': y_val,
            'X_test': X_test, 'y_test': y_test,
            'scaler': scaler,
            'train_df': train_df,
            'val_df': val_df,
            'test_df': test_df,
            'df': pd.concat([train_df, val_df, test_df], ignore_index=True)
        }
        
    except Exception as e:
        print(f"\n❌ Error loading Keras preprocessed data: {e}")
        return None


def load_preprocessed_sequences_pytorch(input_dir=PREPROCESSED_DIR):
    """Load preprocessed sequences for PyTorch backend"""
    
    print("\n" + "="*80)
    print("📂 LOADING PYTORCH PREPROCESSED DATA")
    print("="*80)
    
    try:
        # 1. Load config first
        config = joblib.load(f'{input_dir}/config.pkl')
        print(f"✓ Loaded config")
        
        # 2. Load scaler
        scaler = joblib.load(f'{input_dir}/scaler.pkl')
        print(f"✓ Loaded scaler ({scaler.n_features_in_} features)")
        
        # 3. Load PyTorch datasets
        print("Loading PyTorch datasets...")
        train_ds = torch.load(f'{input_dir}/train_dataset.pt')
        val_ds = torch.load(f'{input_dir}/val_dataset.pt')
        test_ds = torch.load(f'{input_dir}/test_dataset.pt')
        
        print(f"✓ Loaded datasets:")
        print(f"   Train: {len(train_ds)} sequences")
        print(f"   Val: {len(val_ds)} sequences")
        print(f"   Test: {len(test_ds)} sequences")
        
        # 4. Update global variables
        global SEQ_LEN, HORIZON_FRAMES, HORIZON_SECONDS, N_ROWS, N_COLS
        global NUM_ZONES, FEATURE_COLS, CO_ORDINATES, BATCH_SIZE, BACKEND
        
        SEQ_LEN = config['seq_len']
        HORIZON_FRAMES = config['horizon_frames']
        HORIZON_SECONDS = config['horizon_seconds']
        N_ROWS = config['n_rows']
        N_COLS = config['n_cols']
        NUM_ZONES = N_ROWS * N_COLS
        FEATURE_COLS = config['feature_cols']
        CO_ORDINATES = config['coordinate_targets']
        BATCH_SIZE = config['batch_size']
        BACKEND = config.get('backend', 'torch')
        
        print(f"\n✓ Configuration updated:")
        print(f"   SEQ_LEN={SEQ_LEN}, HORIZON={HORIZON_FRAMES} frames ({HORIZON_SECONDS}s)")
        print(f"   Grid: {N_ROWS}x{N_COLS}={NUM_ZONES} zones")
        print(f"   Features: {len(FEATURE_COLS)}")
        print(f"   Coordinate targets: {CO_ORDINATES}")
        
        print("\n" + "="*80)
        print("✅ PYTORCH PREPROCESSED DATA LOADED SUCCESSFULLY")
        print("="*80)
        
        return {
            'train_ds': train_ds,
            'val_ds': val_ds,
            'test_ds': test_ds,
            'scaler': scaler
        }
        
    except Exception as e:
        print(f"\n❌ Error loading PyTorch preprocessed data: {e}")
        return None


# ============================================================================
# MAIN DECISION LOGIC
# ============================================================================

print("\n🔍 Checking for preprocessed sequences...")
print(f"Backend: {BACKEND}")

preprocessed_available, message = check_preprocessed_exists(backend=BACKEND)

if preprocessed_available and USE_PREPROCESSED:
    print(f"✅ {message}")
    print(f"📂 Loading from: {PREPROCESSED_DIR}/")
    
    # Load based on backend
    if BACKEND == "keras":
        loaded_data = load_preprocessed_sequences_keras(PREPROCESSED_DIR)
        
        if loaded_data is not None:
            # Extract all variables
            X_train = loaded_data['X_train']
            y_train = loaded_data['y_train']
            X_val = loaded_data['X_val']
            y_val = loaded_data['y_val']
            X_test = loaded_data['X_test']
            y_test = loaded_data['y_test']
            scaler = loaded_data['scaler']
            train_df = loaded_data['train_df']
            val_df = loaded_data['val_df']
            test_df = loaded_data['test_df']
            df = loaded_data['df']
            
            SKIP_DATA_LOADING = True
            print("✅ Keras data loaded into memory")
            
        else:
            print("⚠️  Failed to load preprocessed Keras data. Will load raw data instead.")
            SKIP_DATA_LOADING = False
    
    elif BACKEND == "torch":
        loaded_data = load_preprocessed_sequences_pytorch(PREPROCESSED_DIR)
        
        if loaded_data is not None:
            # Extract PyTorch datasets
            train_ds = loaded_data['train_ds']
            val_ds = loaded_data['val_ds']
            test_ds = loaded_data['test_ds']
            scaler = loaded_data['scaler']
            
            SKIP_DATA_LOADING = True
            print("✅ PyTorch datasets loaded into memory")
            
        else:
            print("⚠️  Failed to load preprocessed PyTorch data. Will load raw data instead.")
            SKIP_DATA_LOADING = False
        
else:
    if not preprocessed_available:
        print(f"❌ {message}")
    else:
        print("⚠️  USE_PREPROCESSED = False, will load raw data")
    
    print(f"\n📥 Will proceed with normal data loading and preprocessing for {BACKEND}...")
    print("   (This may take 5-10 minutes)")
    SKIP_DATA_LOADING = False



🔍 Checking for preprocessed sequences...
Backend: keras
❌ Preprocessed directory not found: /kaggle/input/preprocessed-sequences/

📥 Will proceed with normal data loading and preprocessing for keras...
   (This may take 5-10 minutes)


## Loading Dataset

In [21]:
if not SKIP_DATA_LOADING:
    print("Loading dataset...")
    
    try:
        df = pd.read_csv(file_path)
        print(f"Dataset loaded successfully! Shape: {df.shape}")
    
        # Display basic info about the dataset
        print(f"Columns: {list(df.columns)}")
        print(f"Unique matches: {df['match_id'].nunique()}")
        print(f"Unique players: {df['player_id'].nunique()}")
    
        if 'timestamp_seconds' in df.columns:
            print(f"Time range: {df['timestamp_seconds'].min():.2f} to {df['timestamp_seconds'].max():.2f} seconds")
    
    except FileNotFoundError:
        print(f"Error: Could not find {file_path}")
        print("Please ensure the combined_matches.csv file exists in the csv_labeled_data folder")
        raise
    
    # Display all columns
    print("\nAll columns in the DataFrame:")
    for col in df.columns:
        print(f"- {col}")
    
    
    # Add enhanced features
    print("Adding velocity and enhanced features...")
    df = add_velocity_features(df)
    
    # Add contextual features if available
    if hasattr(globals(), 'add_contextual_features'):
        df = add_contextual_features(df)
    
    # Check for required columns and handle missing values
    print("Checking feature columns...")
    available_features = [col for col in FEATURE_COLS if col in df.columns]
    missing_features = [col for col in FEATURE_COLS if col not in df.columns]
    
    if missing_features:
        print(f"Warning: Missing columns {missing_features}")
        print(f"Available features: {available_features}")
    
        # Use only available features
        FEATURE_COLS = available_features
        print(f"Updated feature list to {len(FEATURE_COLS)} available features")
    
    # Handle missing values
    print("Handling missing values...")
    initial_rows = len(df)
    df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
    final_rows = len(df)
    print(f"Removed {initial_rows - final_rows} rows with missing values")
    
    print(f"Dataset processed successfully! Final shape: {df.shape}")
    
    # Show data distribution
    print("\nData distribution by match:")
    match_counts = df['match_id'].value_counts().sort_index()
    print(match_counts)
    
    print("\nSample of processed data:")
    print(df[['match_id', 'player_id', 'frame_number'] + FEATURE_COLS[:5]].head())

    
else:
    print("✅ Skipping data loading - using preprocessed sequences")
    print(f"   Dataset shape: {df.shape}")
    print(f"   Features already computed: {len(FEATURE_COLS)}")

Loading dataset...
Dataset loaded successfully! Shape: (5330304, 52)
Columns: ['match_id', 'frame_number', 'period', 'timestamp_seconds', 'time_string', 'trackable_object', 'object_type', 'player_id', 'team_id', 'team_name', 'team_type', 'player_name', 'player_number', 'position_raw', 'position_category', 'x', 'y', 'field_zone', 'penalty_area', 'distance_from_center', 'distance_to_ball', 'speed', 'acceleration', 'team_centroid_distance', 'team_spread', 'nearest_opponent_1', 'nearest_opponent_2', 'nearest_opponent_3', 'defensive_line_distance', 'home_team', 'away_team', 'home_score', 'away_score', 'is_goalkeeper', 'is_defender', 'is_midfielder', 'is_forward', 'x_normalized', 'y_normalized', 'speed_normalized', 'ball_possession_proximity', 'defensive_third', 'attacking_third', 'wide_position', 'possession_team', 'possession_player_name', 'possession_player_id', 'possession_position', 'has_possession', 'team_has_possession', 'opponent_has_possession', 'match_number']
Unique matches: 9
Uni

## Match specific Data inspection

In [22]:
# # 1. Setup display for all columns
# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', 1000)

# # 2. Filter for match_id 2068 AND timestamp_seconds > 1
# # Then take the first 10 rows found
# match_2068_filtered = df[(df['match_id'] == 2068) & (df['timestamp_seconds'] > 1.0)].head(10)

# # 3. Display the results
# print(f"\n--- First 10 rows of Match ID 2068 (Time > 1.0s) ---")
# if not match_2068_filtered.empty:
#     print(match_2068_filtered)
    
#     # Quick sanity check on time gap
#     time_diffs = match_2068_filtered['timestamp_seconds'].diff().dropna()
#     print(f"\nAverage time step: {time_diffs.mean():.4f} seconds")
# else:
#     print("No data found for Match 2068 with timestamp > 1.0s.")

# # Reset display
# pd.reset_option('display.max_columns')

## Data Splitting and Preparation 
### fit scaler on training features

In [23]:
# REPLACE "Data Splitting and Preparation" cell with:

if not SKIP_DATA_LOADING:
    print("Splitting data by matches...")
    train_df, val_df, test_df = split_by_match_df(df)
    
    print("Fitting scaler on training data...")
    scaler = StandardScaler()
    scaler.fit(train_df[FEATURE_COLS].values)
else:
    print("✅ Skipping data splitting - using preprocessed splits")
    print(f"   Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    print(f"   Scaler ready with {scaler.n_features_in_} features")

Splitting data by matches...
Split matches: 6 1 2
Rows per split: 3387911 699902 929216
Fitting scaler on training data...


## SAVE PREPROCESSED SEQUENCES FOR FAST RELOADING


In [ ]:

def check_memory_usage(threshold=80):
    """Check if memory usage exceeds threshold (%)"""
    memory = psutil.virtual_memory()
    usage_percent = memory.percent
    if usage_percent > threshold:
        print(f"\n⚠️  MEMORY WARNING: {usage_percent:.1f}% used (threshold: {threshold}%)")
        return True
    return False

def save_preprocessed_sequences(output_dir='preprocessed_data', chunk_size=500, memory_threshold=80):
    """Save preprocessed sequences with memory monitoring"""
    
    if SKIP_DATA_LOADING:
        print("⚠️  Data already loaded from preprocessed files. Skipping save.")
        return
    
    os.makedirs(output_dir, exist_ok=True)
    
    print("\n💾 Saving preprocessed sequences (memory-efficient mode)...")
    print(f"Backend: {BACKEND}")
    print(f"Memory threshold: {memory_threshold}%")
    
    # 1. Save scaler
    joblib.dump(scaler, f'{output_dir}/scaler.pkl')
    print(f"✓ Saved scaler")
    
    # 2. Save config
    config = {
        'seq_len': SEQ_LEN,
        'horizon_frames': HORIZON_FRAMES,
        'horizon_seconds': HORIZON_SECONDS,
        'n_rows': N_ROWS,
        'n_cols': N_COLS,
        'feature_cols': FEATURE_COLS,
        'coordinate_targets': CO_ORDINATES,
        'batch_size': BATCH_SIZE,
        'backend': BACKEND
    }
    joblib.dump(config, f'{output_dir}/config.pkl')
    print(f"✓ Saved config")
    
    if BACKEND == "keras":
        print("Saving Keras sequences with memory monitoring...")
        
        # Save dataframes first
        train_df.to_pickle(f'{output_dir}/train_df.pkl')
        val_df.to_pickle(f'{output_dir}/val_df.pkl')
        test_df.to_pickle(f'{output_dir}/test_df.pkl')
        print(f"✓ Saved dataframe splits")
        
        with h5py.File(f'{output_dir}/sequences.h5', 'w') as f:
            for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
                print(f"\nProcessing {split_name} split...")
                
                # Check memory before starting
                if check_memory_usage(memory_threshold):
                    print(f"❌ Memory threshold exceeded before {split_name}. Stopping save.")
                    return False
                
                gen = keras_sequence_generator(
                    split_df, FEATURE_COLS, scaler, SEQ_LEN, HORIZON_FRAMES,
                    coordinate_targets=CO_ORDINATES, n_rows=N_ROWS, n_cols=N_COLS
                )
                
                X_dataset = None
                y_dataset = None
                total_sequences = 0
                batch = []
                
                for seq, target in gen:
                    # Check memory every chunk
                    if len(batch) > 0 and len(batch) % chunk_size == 0:
                        if check_memory_usage(memory_threshold):
                            print(f"\n❌ Memory threshold exceeded at {total_sequences} sequences for {split_name}")
                            print(f"   Partial data saved. Consider:")
                            print(f"   1. Reducing chunk_size (current: {chunk_size})")
                            print(f"   2. Increasing memory_threshold (current: {memory_threshold}%)")
                            print(f"   3. Using smaller dataset or more RAM")
                            return False
                    
                    batch.append((seq, target))
                    
                    if len(batch) >= chunk_size:
                        X_chunk = np.array([b[0] for b in batch], dtype=np.float32)
                        y_chunk = np.array([b[1] for b in batch], dtype=np.float32 if CO_ORDINATES else np.int32)
                        
                        if X_dataset is None:
                            y_shape = (0, 2) if CO_ORDINATES else (0,)
                            y_maxshape = (None, 2) if CO_ORDINATES else (None,)
                            
                            X_dataset = f.create_dataset(
                                f'X_{split_name}',
                                shape=(0, SEQ_LEN, len(FEATURE_COLS)),
                                maxshape=(None, SEQ_LEN, len(FEATURE_COLS)),
                                dtype=np.float32,
                                compression='gzip',
                                compression_opts=4
                            )
                            
                            y_dataset = f.create_dataset(
                                f'y_{split_name}',
                                shape=y_shape,
                                maxshape=y_maxshape,
                                dtype=np.float32 if CO_ORDINATES else np.int32,
                                compression='gzip',
                                compression_opts=4
                            )
                        
                        X_dataset.resize((total_sequences + len(X_chunk), SEQ_LEN, len(FEATURE_COLS)))
                        X_dataset[total_sequences:total_sequences+len(X_chunk)] = X_chunk
                        
                        y_dataset.resize((total_sequences + len(y_chunk),) + y_chunk.shape[1:])
                        y_dataset[total_sequences:total_sequences+len(y_chunk)] = y_chunk
                        
                        total_sequences += len(X_chunk)
                        
                        mem_percent = psutil.virtual_memory().percent
                        print(f"  {split_name}: {total_sequences:,} sequences | RAM: {mem_percent:.1f}%", end='\r')
                        
                        batch = []
                        del X_chunk, y_chunk
                        import gc
                        gc.collect()
                
                # Save remaining
                if batch:
                    X_chunk = np.array([b[0] for b in batch], dtype=np.float32)
                    y_chunk = np.array([b[1] for b in batch], dtype=np.float32 if CO_ORDINATES else np.int32)
                    
                    X_dataset.resize((total_sequences + len(X_chunk), SEQ_LEN, len(FEATURE_COLS)))
                    X_dataset[total_sequences:total_sequences+len(X_chunk)] = X_chunk
                    
                    y_dataset.resize((total_sequences + len(y_chunk),) + y_chunk.shape[1:])
                    y_dataset[total_sequences:total_sequences+len(y_chunk)] = y_chunk
                    
                    total_sequences += len(X_chunk)
                    del X_chunk, y_chunk, batch
                    import gc
                    gc.collect()
                
                print(f"\n✓ Saved {split_name}: {total_sequences:,} sequences")
        
        print(f"\n✅ All Keras sequences saved successfully")
        return True
    
    elif BACKEND == "torch":
        print("Saving PyTorch datasets...")
        
        torch.save(train_ds, f'{output_dir}/train_dataset.pt')
        if check_memory_usage(memory_threshold):
            print(f"⚠️  High memory usage after saving train dataset")
        
        torch.save(val_ds, f'{output_dir}/val_dataset.pt')
        if check_memory_usage(memory_threshold):
            print(f"⚠️  High memory usage after saving val dataset")
        
        torch.save(test_ds, f'{output_dir}/test_dataset.pt')
        
        print(f"✓ Saved PyTorch datasets")
        return True
    
    print(f"\n✅ All data saved to '{output_dir}/'")
    return True

# Call with memory monitoring
# success = save_preprocessed_sequences(chunk_size=200, memory_threshold=80)
# if not success:
#     print("\n⚠️  Saving stopped due to memory constraints")

In [24]:
def save_preprocessed_sequences(output_dir='preprocessed_data', chunk_size=1000):
    """Save preprocessed sequences in chunks to avoid memory overflow"""
    
    if SKIP_DATA_LOADING:
        print("⚠️  Data already loaded from preprocessed files. Skipping save.")
        return
    
    os.makedirs(output_dir, exist_ok=True)
    
    print("\n💾 Saving preprocessed sequences (memory-efficient mode)...")
    print(f"Backend: {BACKEND}")
    
    # 1. Save scaler (common to both)
    joblib.dump(scaler, f'{output_dir}/scaler.pkl')
    print(f"✓ Saved scaler")
    
    # 2. Save config (common to both)
    config = {
        'seq_len': SEQ_LEN,
        'horizon_frames': HORIZON_FRAMES,
        'horizon_seconds': HORIZON_SECONDS,
        'n_rows': N_ROWS,
        'n_cols': N_COLS,
        'feature_cols': FEATURE_COLS,
        'coordinate_targets': CO_ORDINATES,
        'batch_size': BATCH_SIZE,
        'backend': BACKEND
    }
    joblib.dump(config, f'{output_dir}/config.pkl')
    print(f"✓ Saved config")
    
    # 3. Backend-specific saving
    if BACKEND == "keras":
        print("Saving Keras sequences WITH GENERATOR (no memory buildup)...")
        
        # Save dataframe splits first (small)
        train_df.to_pickle(f'{output_dir}/train_df.pkl')
        val_df.to_pickle(f'{output_dir}/val_df.pkl')
        test_df.to_pickle(f'{output_dir}/test_df.pkl')
        print(f"✓ Saved dataframe splits")
        
        # Create HDF5 file with incremental writing
        with h5py.File(f'{output_dir}/sequences.h5', 'w') as f:
            for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
                print(f"\nProcessing {split_name} split (generator mode)...")
                
                # Get generator
                gen = keras_sequence_generator(
                    split_df, FEATURE_COLS, scaler, SEQ_LEN, HORIZON_FRAMES,
                    coordinate_targets=CO_ORDINATES, n_rows=N_ROWS, n_cols=N_COLS
                )
                
                # Initialize datasets
                X_dataset = None
                y_dataset = None
                total_sequences = 0
                batch = []
                
                # Process generator in batches
                for seq, target in gen:
                    batch.append((seq, target))
                    
                    if len(batch) >= chunk_size:
                        # Convert batch to arrays
                        X_chunk = np.array([b[0] for b in batch], dtype=np.float32)
                        y_chunk = np.array([b[1] for b in batch], dtype=np.float32 if CO_ORDINATES else np.int32)
                        
                        # Initialize datasets on first batch
                        if X_dataset is None:
                            y_shape = (0, 2) if CO_ORDINATES else (0,)
                            y_maxshape = (None, 2) if CO_ORDINATES else (None,)
                            
                            X_dataset = f.create_dataset(
                                f'X_{split_name}',
                                shape=(0, SEQ_LEN, len(FEATURE_COLS)),
                                maxshape=(None, SEQ_LEN, len(FEATURE_COLS)),
                                dtype=np.float32,
                                compression='gzip',
                                compression_opts=4
                            )
                            
                            y_dataset = f.create_dataset(
                                f'y_{split_name}',
                                shape=y_shape,
                                maxshape=y_maxshape,
                                dtype=np.float32 if CO_ORDINATES else np.int32,
                                compression='gzip',
                                compression_opts=4
                            )
                        
                        # Append to HDF5
                        X_dataset.resize((total_sequences + len(X_chunk), SEQ_LEN, len(FEATURE_COLS)))
                        X_dataset[total_sequences:total_sequences+len(X_chunk)] = X_chunk
                        
                        y_dataset.resize((total_sequences + len(y_chunk),) + y_chunk.shape[1:])
                        y_dataset[total_sequences:total_sequences+len(y_chunk)] = y_chunk
                        
                        total_sequences += len(X_chunk)
                        print(f"  {split_name}: {total_sequences:,} sequences saved...", end='\r')
                        
                        # Clear batch and force garbage collection
                        batch = []
                        del X_chunk, y_chunk
                        import gc
                        gc.collect()
                
                # Save remaining sequences
                if batch:
                    X_chunk = np.array([b[0] for b in batch], dtype=np.float32)
                    y_chunk = np.array([b[1] for b in batch], dtype=np.float32 if CO_ORDINATES else np.int32)
                    
                    X_dataset.resize((total_sequences + len(X_chunk), SEQ_LEN, len(FEATURE_COLS)))
                    X_dataset[total_sequences:total_sequences+len(X_chunk)] = X_chunk
                    
                    y_dataset.resize((total_sequences + len(y_chunk),) + y_chunk.shape[1:])
                    y_dataset[total_sequences:total_sequences+len(y_chunk)] = y_chunk
                    
                    total_sequences += len(X_chunk)
                    del X_chunk, y_chunk, batch
                    import gc
                    gc.collect()
                
                print(f"\n✓ Saved {split_name}: {total_sequences:,} sequences")
        
        print(f"\n✓ All Keras sequences saved to HDF5")
    
    elif BACKEND == "torch":
        print("Saving PyTorch datasets...")
        torch.save(train_ds, f'{output_dir}/train_dataset.pt')
        torch.save(val_ds, f'{output_dir}/val_dataset.pt')
        torch.save(test_ds, f'{output_dir}/test_dataset.pt')
        
        print(f"✓ Saved PyTorch datasets:")
        print(f"   Train: {len(train_ds)} sequences")
        print(f"   Val: {len(val_ds)} sequences")
        print(f"   Test: {len(test_ds)} sequences")
    
    print(f"\n✅ All {BACKEND} data saved to '{output_dir}/' directory")
    print(f"   Next time, set PREPROCESSED_DIR='{output_dir}' to load instantly!")

# Call with smaller chunk size for 30GB RAM
# save_preprocessed_sequences(chunk_size=500)  # Reduced from 1000


💾 Saving preprocessed sequences (memory-efficient mode)...
Backend: keras
✓ Saved scaler
✓ Saved config
Saving Keras sequences WITH GENERATOR (no memory buildup)...
✓ Saved dataframe splits

Processing train split (generator mode)...
  train: 3,362,000 sequences saved...
✓ Saved train: 3,362,134 sequences

Processing val split (generator mode)...
  val: 696,000 sequences saved...
✓ Saved val: 696,028 sequences

Processing test split (generator mode)...
  test: 921,000 sequences saved...
✓ Saved test: 921,170 sequences

✓ All Keras sequences saved to HDF5

✅ All keras data saved to 'preprocessed_data/' directory
   Next time, set PREPROCESSED_DIR='preprocessed_data' to load instantly!


## pytorch dataset initialisation into data sequence and loaders

In [25]:
if BACKEND == "torch":
    if not SKIP_DATA_LOADING:
        print("Creating PyTorch sequence datasets...")

        train_ds = FootballSequenceDataset(
            train_df,
            seq_len=SEQ_LEN,
            horizon=HORIZON_FRAMES,
            feature_cols=FEATURE_COLS,
            scaler=scaler
        )

        val_ds = FootballSequenceDataset(
            val_df,
            seq_len=SEQ_LEN,
            horizon=HORIZON_FRAMES,
            feature_cols=FEATURE_COLS,
            scaler=scaler
        )

        test_ds = FootballSequenceDataset(
            test_df,
            seq_len=SEQ_LEN,
            horizon=HORIZON_FRAMES,
            feature_cols=FEATURE_COLS,
            scaler=scaler
        )

        print(f"Dataset sizes - Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
    else:
        print("✅ Using preprocessed PyTorch datasets")
        print(f"   Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

    # Create DataLoaders (always needed)
    print("Creating PyTorch DataLoaders...")
    
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=worker_num,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        num_workers=worker_num,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        num_workers=worker_num,
        pin_memory=True
    )
    
    print(f"✓ DataLoaders created with batch_size={BATCH_SIZE}")

## Keras dataset initialisation into data sequence and loaders

In [ ]:

if BACKEND == "keras":
    # Check available GPUs
    gpus = tf.config.list_physical_devices('GPU')
    print(f"\n🔍 GPU Detection:")
    print(f"Available GPUs: {len(gpus)}")
    for gpu in gpus:
        print(f"  - {gpu}")
    
    # Create distribution strategy for multi-GPU
    if len(gpus) > 1:
        print(f"\n🚀 Using MirroredStrategy for {len(gpus)} GPUs")
        strategy = tf.distribute.MirroredStrategy()
    else:
        print("\n⚠️  Single/No GPU detected, using default strategy")
        strategy = tf.distribute.get_strategy()
    
    print(f"Number of devices in strategy: {strategy.num_replicas_in_sync}")
    
    # Adjust batch size for multi-GPU (IMPORTANT!)
    GLOBAL_BATCH_SIZE = BATCH_SIZE * strategy.num_replicas_in_sync
    print(f"\nBatch size configuration:")
    print(f"  • Per-GPU batch size: {BATCH_SIZE}")
    print(f"  • Global batch size: {GLOBAL_BATCH_SIZE}")
else:
    print(f"\n🔥 PyTorch backend - multi-GPU handled differently")
    strategy = None  # Not needed for PyTorch

In [ ]:
if BACKEND == "keras":
       if not SKIP_DATA_LOADING:
        print("Creating Keras sequence datasets via tf.data...")
        
        train_loader = make_tf_dataset(
            train_df, 
            GLOBAL_BATCH_SIZE, 
            shuffle=True, 
            coordinate_targets=CO_ORDINATES,
        )
        
        val_loader = make_tf_dataset(val_df, GLOBAL_BATCH_SIZE, shuffle=False, coordinate_targets=CO_ORDINATES)
        test_loader = make_tf_dataset(test_df, GLOBAL_BATCH_SIZE, shuffle=False, coordinate_targets=CO_ORDINATES)
        
        print(f"Datasets created. Ready for training.")
       else:
            print("✅ Using preprocessed sequences - creating tf.data.Datasets...")
            
            # Create datasets from arrays
            train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
            train_loader = train_dataset.shuffle(2048).batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
            
            val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
            val_loader = val_dataset.batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
            
            test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
            test_loader = test_dataset.batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
            
            print(f"✓ Train batches: {len(train_loader)}")
            print(f"✓ Val batches: {len(val_loader)}")
            print(f"✓ Test batches: {len(test_loader)}")

Creating Keras sequence datasets via tf.data...
Instructions for updating:
Use output_signature instead
Instructions for updating:
Use output_signature instead
Datasets created. Ready for training.


I0000 00:00:1770055537.209015     164 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1770055537.215173     164 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [27]:
# if BACKEND == "keras":
#     print("Creating Keras sequence datasets...")

    
#     if CO_ORDINATES:
#         X_train, y_train = build_keras_sequences(
#             train_df, FEATURE_COLS, scaler, SEQ_LEN, HORIZON_FRAMES, coordinate_targets=True)
#         X_val, y_val = build_keras_sequences(
#             val_df, FEATURE_COLS, scaler, SEQ_LEN, HORIZON_FRAMES, coordinate_targets=True)
#         X_test, y_test = build_keras_sequences(
#             test_df, FEATURE_COLS, scaler, SEQ_LEN, HORIZON_FRAMES, coordinate_targets=True)
#     else:
#         X_train, y_train = build_keras_sequences(
#             train_df, FEATURE_COLS, scaler, SEQ_LEN, HORIZON_FRAMES,
#             coordinate_targets=False, n_rows=N_ROWS, n_cols=N_COLS)
#         X_val, y_val = build_keras_sequences(
#             val_df, FEATURE_COLS, scaler, SEQ_LEN, HORIZON_FRAMES,
#             coordinate_targets=False, n_rows=N_ROWS, n_cols=N_COLS)
#         X_test, y_test = build_keras_sequences(
#             test_df, FEATURE_COLS, scaler, SEQ_LEN, HORIZON_FRAMES,
#             coordinate_targets=False, n_rows=N_ROWS, n_cols=N_COLS)
       
#  # Create tf.data.Dataset using your function
#     train_loader = make_tf_dataset(train_df, BATCH_SIZE, shuffle=True, coordinate_targets=CO_ORDINATES)
#     val_loader   = make_tf_dataset(val_df, BATCH_SIZE, shuffle=True, coordinate_targets=CO_ORDINATES)
#     test_loader  = make_tf_dataset(test_df, BATCH_SIZE, shuffle=True, coordinate_targets=CO_ORDINATES)

#     print(f"Keras dataset sizes - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
#     print(f"Dataset sizes - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")


In [28]:
import tensorflow as tf
tf.keras.backend.clear_session()
import gc
gc.collect()

0

## Model Initialisation

In [29]:
# Initialize enhanced model
model_type = MODEL_TYPE  # Change to "tcn" for Temporal Convolutional Network
print(f"Initializing {MODEL_TYPE}")

if model_type == "enhanced":
    model = EnhancedFootballModel(
        input_dim=len(FEATURE_COLS),
        num_classes=NUM_ZONES,
        hidden_dim=128,
        num_layers=2,
        dropout=0.3
    ).to(device)
    
if model_type == "tcn":
    model = TemporalConvNet(
        input_dim=len(FEATURE_COLS),
        num_classes=NUM_ZONES,
        num_channels=[64, 128, 256],
        dropout=0.3
    ).to(device)
    
if model_type == "tcn_new":
    model = ZoneTCN(input_dim=len(FEATURE_COLS),num_classes=NUM_ZONES,channels=[64, 128, 256, 256, 256, 256],dropout=0.3).to(device)

if model_type == "LocusLab_tcn":
    model = DeltaTCN(
        input_dim=len(FEATURE_COLS),
        num_classes=2,
        channels=[64, 128, 256, 512],  # Increased capacity slightly
        kernel_size=5,                 # Larger kernel often helps with movement
        dropout=0.1                    # Reduced dropout to help underfitting
    ).to(device)

if model_type == "Keras_tcn":
    with strategy.scope():
        model = compiled_tcn(
            num_feat=len(FEATURE_COLS),
            num_classes = 2 if CO_ORDINATES else NUM_ZONES,    # or 2 for dx, dy
            nb_filters=256,   # matches channels
            kernel_size=5,
            dilations=[1, 2, 4, 8, 16, 32,64],     # SAME as PyTorch
            nb_stacks=2,               # PyTorch uses 1 stack
            max_len=SEQ_LEN,              # dynamic sequence length
            padding='causal',          # replaces Chomp1d
            use_skip_connections=True,# VERY IMPORTANT
            return_sequences=False,    # take last timestep only
            dropout_rate=0.2,
            activation='relu',
            use_batch_norm=True,
            use_layer_norm=False,
            lr=LR,
            regression=CO_ORDINATES )  # True = dx/dy, False = zones
        if CO_ORDINATES:
            model.compile(optimizer=tf.keras.optimizers
                          .Adam(learning_rate=5e-5,clipnorm=1.0),
                          loss='mse',
                          metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

if BACKEND == "pytorch":
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs for PyTorch model")
        model = nn.DataParallel(model)
    model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

elif BACKEND == "keras":
    print("Keras model, no DataParallel needed")
    # Keras automatically uses available GPU
    total_params = model.count_params()
    trainable_params = sum([np.prod(w.shape) for w in model.trainable_weights])

print(f"Model initialized: {model_type}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Input features: {len(FEATURE_COLS)}")
print(f"Output size: {2 if CO_ORDINATES else NUM_ZONES}")



Initializing Keras_tcn
x.shape= (None, 256)
model.x = (None, 100, 28)
model.y = (None, 2)
Keras model, no DataParallel needed
Model initialized: Keras_tcn
Total parameters: 8,926,978
Trainable parameters: 8,912,642
Input features: 28
Output size: 2


##  FORWARD PASS TEST (PyTorch + Keras)


In [30]:

if BACKEND == "pytorch":
    print("Testing model with sample data...")
    sample_batch = next(iter(train_loader))
    sample_input, sample_target = sample_batch
    sample_input = sample_input.to(device)
    with torch.no_grad():
        sample_output = model(sample_input)
        print(f"Sample input shape: {sample_input.shape}")
        print(f"Sample output shape: {sample_output.shape}")
        print("Model test successful!")

elif BACKEND == "keras":
    print("Testing Keras model with sample data...")

    # sample_input, sample_target from tf.data.Dataset
    sample_input, sample_target = next(iter(train_loader))

    # Keras datasets should already return arrays, but handle dict if any
    if isinstance(sample_target, dict):
        sample_target = list(sample_target.values())[0]

    # Ensure numpy arrays
    sample_input = np.array(sample_input)
    sample_target = np.array(sample_target)

    # Forward pass in Keras
    sample_output = model(sample_input, training=False)  # use __call__, not .predict
    # Alternatively, model.predict(sample_input, batch_size=len(sample_input), verbose=0) works if model is pure Keras

    print(f"Sample input shape: {sample_input.shape}")
    print(f"Sample output shape: {sample_output.shape}")
    print("Keras model test successful!")

Testing Keras model with sample data...


I0000 00:00:1770055543.464309     164 cuda_dnn.cc:529] Loaded cuDNN version 91002


Sample input shape: (16, 100, 28)
Sample output shape: (16, 2)
Keras model test successful!


In [31]:
# import numpy as np
# import pandas as pd

# def inspect_keras_predictions(model, dataset, num_samples=10, coordinate_targets=False):
#     """
#     Sanity check predictions for Keras model.
    
#     Args:
#         model: trained Keras model
#         dataset: tf.data.Dataset or (X, y) tuple
#         num_samples: number of samples to display
#         coordinate_targets: if True, predicts dx/dy, else zones
#     """
    
#     # --- Get one batch ---
#     if isinstance(dataset, tf.data.Dataset):
#         sample_input, sample_target = next(iter(dataset))
#         sample_input = sample_input.numpy()
#         sample_target = sample_target.numpy()
#     else:
#         sample_input, sample_target = dataset
#         sample_input = np.array(sample_input)
#         sample_target = np.array(sample_target)

#     # Take a small subset
#     sample_input = sample_input[:num_samples]
#     sample_target = sample_target[:num_samples]

#     # --- Forward pass ---
#     sample_output = model(sample_input, training=False)  # use __call__ for Functional API
#     sample_output = sample_output.numpy()

#     if coordinate_targets:
#         # regression: dx/dy
#         results = pd.DataFrame({
#             'True_dx': sample_target[:, 0],
#             'Pred_dx': sample_output[:, 0],
#             'True_dy': sample_target[:, 1],
#             'Pred_dy': sample_output[:, 1],
#         })
#         results['Error_X'] = abs(results['True_dx'] - results['Pred_dx'])
#         results['Error_Y'] = abs(results['True_dy'] - results['Pred_dy'])
#     else:
#         # classification: zones
#         pred_classes = np.argmax(sample_output, axis=1)
#         results = pd.DataFrame({
#             'True_zone': sample_target.astype(int),
#             'Pred_zone': pred_classes
#         })
#         results['Correct'] = results['True_zone'] == results['Pred_zone']

#     print("\n--- SANITY CHECK ---")
#     print(results)
#     print("\nPrediction Statistics:")
#     print(f"Output max: {sample_output.max():.4f}, min: {sample_output.min():.4f}, mean: {sample_output.mean():.4f}")
#     if not coordinate_targets:
#         print(f"Accuracy on this batch: {results['Correct'].mean():.4f}")

# # --- Example usage ---
# if BACKEND == "keras":
#     inspect_keras_predictions(model, (X_val[:32], y_val[:32]), num_samples=10, coordinate_targets=CO_ORDINATES)


NameError: name 'X_val' is not defined

##  MODEL TRAINING 

In [ ]:

print("Starting enhanced training...")
print(f"Training configuration:")
print(f"Training for {'regression (coordinates)' if CO_ORDINATES else 'classification (zones)'}")
print(f"- Epochs: {EPOCHS}")
print(f"- Learning rate: {LR}")
print(f"- Batch size: {BATCH_SIZE}")
print(f"- Patience: {PATIENCE}")
print(f"- Device: {device}")
print(f"-Model Type:{MODEL_TYPE}")
print(f"-Model Type:{BACKEND}")

trained_model, training_history = train_model(model, train_loader, val_loader,epochs=EPOCHS, lr=LR, patience=PATIENCE)
# trained_model, training_history = train_model_enhanced(model, train_loader, val_loader,epochs=EPOCHS, lr=LR, patience=PATIENCE)

print("Training completed successfully!")




Starting enhanced training...
Training configuration:
Training for regression (coordinates)
- Epochs: 20
- Learning rate: 0.0001
- Batch size: 16
- Patience: 2
- Device: cuda
-Model Type:Keras_tcn
-Model Type:keras
Epoch 1/20


I0000 00:00:1770055606.827209     221 service.cc:152] XLA service 0x7d7bd8001d10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770055606.827250     221 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1770055606.827256     221 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
2026-02-02 18:06:58.727717: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-02-02 18:06:59.031077: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


      2/Unknown 48s 86ms/step - loss: 73.8691 - mae: 7.0247

I0000 00:00:1770055634.761518     221 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 101327/Unknown 5029s 49ms/step - loss: 0.8078 - mae: 0.3651

In [ ]:
import os

# Create a specific folder if you want to stay organized
output_dir = '/kaggle/working/models'
os.makedirs(output_dir, exist_ok=True)

model_save_path = os.path.join(output_dir, 'football_tcn_delta_model.keras')

trained_model.save(model_save_path)

print(f"Model saved successfully to: {model_save_path}")

In [ ]:
from tensorflow.keras.models import load_model
import joblib

# You must provide the custom classes in a dictionary
custom_objects = {
    'TCN': TCN,
    'ResidualBlock': ResidualBlock
}

reloaded_model = load_model(model_save_path, custom_objects=custom_objects)
print("Model reloaded successfully!")

scaler_path = os.path.join(output_dir, 'feature_scaler.pkl')
joblib.dump(scaler, scaler_path)
print(f"Scaler saved to: {scaler_path}")

In [ ]:
# ==== PLOT TRAINING HISTORY ====
# Access the actual dictionary inside the History object
history_dict = training_history.history 

plt.figure(figsize=(15, 5))

# --- 1. Training & Validation Loss (regression) ---
plt.subplot(1, 2, 1)
# Use the dictionary to get the values. 
# Note: Keras usually uses 'loss' and 'val_loss' (not 'train_loss')
plt.plot(history_dict.get('loss', []), label='Train Loss', color='blue')

val_loss = history_dict.get('val_loss', None)
if val_loss is not None:
    plt.plot(val_loss, label='Val Loss', color='red')

plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)

# --- 2. MAE Metrics (Since you are doing Coordinates) ---
if CO_ORDINATES:
    plt.subplot(1, 2, 2)
    plt.plot(history_dict.get('mae', []), label='Train MAE', color='blue')
    plt.plot(history_dict.get('val_mae', []), label='Val MAE', color='red')
    plt.title('Mean Absolute Error (Pitch Units)')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.legend()
    plt.grid(True)

In [ ]:
# # ==== PLOT TRAINING HISTORY ====
# plt.figure(figsize=(15, 5))

# # --- 1. Training & Validation Loss (regression) ---
# plt.subplot(1, 2, 1)
# plt.plot(training_history.get('train_loss', []), label='Train Loss', color='blue')

# # Plot validation loss if it exists
# val_loss = training_history.get('val_loss', None)
# if val_loss is not None:
#     plt.plot(val_loss, label='Val Loss', color='red')

# plt.title('Training Loss' if CO_ORDINATES else 'Loss / Accuracy')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.legend()
# plt.grid(True)

# # --- 2. Accuracy metrics (classification only) ---
# if not CO_ORDINATES:
#     plt.subplot(1, 2, 2)

#     # Safely get lists and ignore None values
#     train_acc = [v for v in training_history.get('train_acc', []) if v is not None]
#     val_acc   = [v for v in training_history.get('val_acc', []) if v is not None]
#     val_top3  = [v for v in training_history.get('val_top3_acc', []) if v is not None]

#     plt.plot(train_acc, label='Train Accuracy', color='blue')
#     plt.plot(val_acc, label='Validation Accuracy', color='red')
#     if val_top3:
#         plt.plot(val_top3, label='Validation Top-3 Accuracy', color='green')

#     plt.title('Accuracy Metrics')
#     plt.xlabel('Epoch')
#     plt.ylabel('Accuracy')
#     plt.legend()
#     plt.grid(True)

# plt.tight_layout()
# plt.show()

# # ==== PRINT FINAL METRICS ====
# if CO_ORDINATES:
#     if 'val_loss' in training_history and training_history['val_loss']:
#         print(f"\nFinal Validation Loss (MSE): {training_history['val_loss'][-1]:.4f}")
# else:
#     if 'val_acc' in training_history and training_history['val_acc']:
#         print(f"\nFinal Validation Accuracy: {training_history['val_acc'][-1]:.4f}")
#     if 'val_top3_acc' in training_history and training_history['val_top3_acc']:
#         print(f"Final Validation Top-3 Accuracy: {training_history['val_top3_acc'][-1]:.4f}")


In [ ]:
import pandas as pd
import numpy as np
import torch
import tensorflow as tf

def inspect_predictions_backend(model, val_loader, backend='pytorch', top_k=3):
    """
    Inspect predictions on first batch of validation data.
    Works for PyTorch and Keras models.
    """
    # Get first batch
    sample_input, sample_target = next(iter(val_loader))

    # --- Handle PyTorch ---
    if backend == 'pytorch':
        model.eval()
        device = 'cuda' if torch.cuda.is_available() else 'cpu'

        # Targets dict or tensor
        if isinstance(sample_target, dict):
            target_tensor = sample_target['delta'] if CO_ORDINATES else sample_target['zone']
        else:
            target_tensor = sample_target

        sample_input = sample_input.to(device)
        target_tensor = target_tensor.to(device)

        with torch.no_grad():
            preds = model(sample_input)

        if CO_ORDINATES:
            preds_np = preds.cpu().numpy()
            targets_np = target_tensor.cpu().numpy()
            results = pd.DataFrame({
                'True_dx': targets_np[:10, 0],
                'Pred_dx': preds_np[:10, 0],
                'True_dy': targets_np[:10, 1],
                'Pred_dy': preds_np[:10, 1]
            })
            results['Error_X'] = abs(results['True_dx'] - results['Pred_dx'])
            results['Error_Y'] = abs(results['True_dy'] - results['Pred_dy'])
        else:
            preds_class = preds.argmax(dim=1).cpu().numpy()
            targets_np = target_tensor.cpu().numpy()
            results = pd.DataFrame({
                'True_Zone': targets_np[:10],
                'Pred_Zone': preds_class[:10]
            })

    # --- Handle Keras ---
    elif backend == 'keras':
        # Convert tensors to numpy if needed
        if isinstance(sample_input, tf.Tensor):
            sample_input = sample_input.numpy()
        if isinstance(sample_target, tf.Tensor):
            sample_target = sample_target.numpy()
        elif isinstance(sample_target, dict):
            sample_target = list(sample_target.values())[0].numpy()

        preds_np = model(sample_input, training=False).numpy()

        if CO_ORDINATES:
            targets_np = sample_target
            results = pd.DataFrame({
                'True_dx': targets_np[:10, 0],
                'Pred_dx': preds_np[:10, 0],
                'True_dy': targets_np[:10, 1],
                'Pred_dy': preds_np[:10, 1]
            })
            results['Error_X'] = abs(results['True_dx'] - results['Pred_dx'])
            results['Error_Y'] = abs(results['True_dy'] - results['Pred_dy'])
        else:
            preds_class = np.argmax(preds_np, axis=1)
            targets_np = sample_target
            results = pd.DataFrame({
                'True_Zone': targets_np[:10],
                'Pred_Zone': preds_class[:10]
            })

    # Print results
    print("\n--- SANITY CHECK (First 10 Samples) ---")
    print(results.round(4))

    if CO_ORDINATES:
        print("\nPrediction Statistics:")
        print(f"Max Pred Value: {preds_np.max():.4f}")
        print(f"Min Pred Value: {preds_np.min():.4f}")
        print(f"Avg Pred Value: {preds_np.mean():.4f}")
    else:
        if top_k > 1:
            topk_preds = np.argsort(preds_np, axis=1)[:, -top_k:][:10]
            print(f"\nTop-{top_k} predictions for first 10 samples:")
            print(topk_preds)


inspect_predictions_backend(trained_model, val_loader, backend=BACKEND)



In [ ]:
# ==== EVALUATION SECTION ====
print("\n" + "="*80)
print("📊 MODEL EVALUATION ON TEST SET")
print("="*80)

# Ensure output directory exists
output_dir = '/kaggle/working/models'
os.makedirs(output_dir, exist_ok=True)

# Load best model
if BACKEND == "keras":
    try:
        with strategy.scope():
            from tensorflow.keras.models import load_model
            model = load_model(
                f"{output_dir}/best_model_{MODEL_TYPE}.keras",
                custom_objects={'TCN': TCN, 'ResidualBlock': ResidualBlock}
            )
        print(f"✅ Loaded best Keras model")
    except:
        print("⚠️  Using trained model (best model file not found)")
        model = trained_model
else:
    try:
        checkpoint = torch.load(f"{output_dir}/best_model_{MODEL_TYPE}.pth", map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        model.eval()
        print(f"✅ Loaded best PyTorch model")
    except:
        print("⚠️  Using trained model (best model file not found)")
        model = trained_model

# Evaluate model
def evaluate_model_unified(model, test_loader, backend='keras', coordinate_mode=False):
    all_preds = []
    all_labels = []
    
    if backend == 'keras':
        print("Evaluating Keras model...")
        for batch_X, batch_y in test_loader:
            preds = model.predict(batch_X, verbose=0)
            all_preds.append(preds)
            all_labels.append(batch_y.numpy())
        
        all_preds = np.concatenate(all_preds, axis=0)
        all_labels = np.concatenate(all_labels, axis=0)
        
    else:
        print("Evaluating PyTorch model...")
        model.eval()
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                if isinstance(batch_y, dict):
                    batch_y = batch_y['delta'] if coordinate_mode else batch_y['zone']
                preds = model(batch_X)
                all_preds.append(preds.cpu().numpy())
                all_labels.append(batch_y.cpu().numpy() if torch.is_tensor(batch_y) else batch_y)
        
        all_preds = np.concatenate(all_preds, axis=0)
        all_labels = np.concatenate(all_labels, axis=0)
    
    if coordinate_mode:
        # Regression metrics for coordinates
        mae = np.mean(np.abs(all_preds - all_labels))
        mse = np.mean((all_preds - all_labels) ** 2)
        rmse = np.sqrt(mse)
        
        # Per-coordinate metrics
        mae_x = np.mean(np.abs(all_preds[:, 0] - all_labels[:, 0]))
        mae_y = np.mean(np.abs(all_preds[:, 1] - all_labels[:, 1]))
        
        print("\n📊 Coordinate Prediction Metrics:")
        print(f"  Overall MAE: {mae:.4f}")
        print(f"  Overall RMSE: {rmse:.4f}")
        print(f"  X-coordinate MAE: {mae_x:.4f}")
        print(f"  Y-coordinate MAE: {mae_y:.4f}")
        
        return {
            'mae': mae,
            'rmse': rmse,
            'mse': mse,
            'mae_x': mae_x,
            'mae_y': mae_y,
            'predictions': all_preds,
            'labels': all_labels
        }
    else:
        # Classification metrics for zones
        pred_classes = np.argmax(all_preds, axis=1)
        true_classes = np.argmax(all_labels, axis=1) if all_labels.ndim > 1 else all_labels
        
        accuracy = accuracy_score(true_classes, pred_classes)
        
        print("\n📊 Classification Metrics:")
        print(f"  Test Accuracy: {accuracy:.4f}")
        print("\nClassification Report:")
        print(classification_report(true_classes, pred_classes, zero_division=0))
        
        return {
            'accuracy': accuracy,
            'predictions': pred_classes,
            'labels': true_classes,
            'probabilities': all_preds
        }

# Run evaluation
results = evaluate_model_unified(model, test_loader, backend=BACKEND, coordinate_mode=CO_ORDINATES)

# Save predictions
if CO_ORDINATES:
    print(f"\n💾 Saving coordinate predictions...")
    predictions_df = pd.DataFrame({
        'true_x': results['labels'][:, 0],
        'true_y': results['labels'][:, 1],
        'pred_x': results['predictions'][:, 0],
        'pred_y': results['predictions'][:, 1],
        'error_x': np.abs(results['predictions'][:, 0] - results['labels'][:, 0]),
        'error_y': np.abs(results['predictions'][:, 1] - results['labels'][:, 1])
    })
else:
    print(f"\n💾 Saving zone predictions...")
    predictions_df = pd.DataFrame({
        'true_zone': results['labels'],
        'pred_zone': results['predictions'],
        'correct': results['labels'] == results['predictions']
    })

predictions_df.to_csv(f'{output_dir}/test_predictions.csv', index=False)
print(f"✅ Predictions saved to {output_dir}/test_predictions.csv")

print("\n" + "="*80)
print("✅ EVALUATION COMPLETE")
print("="*80)

In [ ]:
# ==== VISUALIZATION OF RESULTS ====
if CO_ORDINATES:
    print("\n📊 Visualizing Coordinate Predictions...")
    
    # Sample for visualization
    sample_size = min(100, len(results['predictions']))
    sample_indices = np.random.choice(len(results['predictions']), sample_size, replace=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Scatter plot: Predicted vs True
    axes[0].scatter(results['labels'][sample_indices, 0], results['predictions'][sample_indices, 0], 
                   alpha=0.5, label='X-coordinate', s=30)
    axes[0].scatter(results['labels'][sample_indices, 1], results['predictions'][sample_indices, 1], 
                   alpha=0.5, label='Y-coordinate', s=30)
    axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect prediction', linewidth=2)
    axes[0].set_xlabel('True Values', fontsize=12)
    axes[0].set_ylabel('Predicted Values', fontsize=12)
    axes[0].set_title('Predicted vs True Coordinates', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Error distribution
    errors = np.sqrt((results['predictions'][:, 0] - results['labels'][:, 0])**2 + 
                     (results['predictions'][:, 1] - results['labels'][:, 1])**2)
    axes[1].hist(errors, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    axes[1].set_xlabel('Euclidean Distance Error', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title(f'Prediction Error Distribution\nMean Error: {errors.mean():.4f}', 
                     fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/prediction_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Visualization saved to {output_dir}/prediction_visualization.png")
else:
    # Confusion matrix for zone classification
    from sklearn.metrics import confusion_matrix
    import seaborn as sns
    
    print("\n📊 Generating Confusion Matrix...")
    cm = confusion_matrix(results['labels'], results['predictions'])
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix - Zone Prediction', fontsize=16, fontweight='bold')
    plt.ylabel('True Zone', fontsize=12)
    plt.xlabel('Predicted Zone', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Confusion matrix saved to {output_dir}/confusion_matrix.png")

In [ ]:
# ==== FINAL SUMMARY ====
print("\n" + "="*80)
print("🏆 TRAINING & EVALUATION SUMMARY")
print("="*80)

print("\n📊 Dataset Information:")
print(f"  • Total sequences: {len(train_df) + len(val_df) + len(test_df):,}")
print(f"  • Training: {len(train_df):,}")
print(f"  • Validation: {len(val_df):,}")
print(f"  • Test: {len(test_df):,}")

print("\n🧠 Model Configuration:")
print(f"  • Model Type: {MODEL_TYPE}")
print(f"  • Backend: {BACKEND}")
print(f"  • Sequence Length: {SEQ_LEN} frames")
print(f"  • Horizon: {HORIZON_SECONDS}s ({HORIZON_FRAMES} frames)")
print(f"  • Features: {len(FEATURE_COLS)}")
print(f"  • Prediction Mode: {'Coordinates (Regression)' if CO_ORDINATES else 'Zones (Classification)'}")

print("\n📈 Training Configuration:")
print(f"  • Batch Size: {BATCH_SIZE}")
if BACKEND == 'keras':
    print(f"  • Global Batch Size (multi-GPU): {GLOBAL_BATCH_SIZE}")
print(f"  • Learning Rate: {LR}")
if BACKEND == 'keras':
    print(f"  • Epochs Completed: {len(training_history.history.get('loss', []))}")
else:
    print(f"  • Epochs Completed: {len(training_history.get('train_loss', []))}")

print("\n🎯 Final Performance:")
if CO_ORDINATES:
    print(f"  • Test MAE: {results['mae']:.4f}")
    print(f"  • Test RMSE: {results['rmse']:.4f}")
    print(f"  • X-coordinate MAE: {results['mae_x']:.4f}")
    print(f"  • Y-coordinate MAE: {results['mae_y']:.4f}")
else:
    print(f"  • Test Accuracy: {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)")

print("\n💾 Saved Files:")
print(f"  • Model: {output_dir}/best_model_{MODEL_TYPE}.keras")
print(f"  • Predictions: {output_dir}/test_predictions.csv")
if CO_ORDINATES:
    print(f"  • Visualization: {output_dir}/prediction_visualization.png")
else:
    print(f"  • Confusion Matrix: {output_dir}/confusion_matrix.png")

print("\n" + "="*80)
print("✅ ALL PROCESSING COMPLETE!")
print("="*80)

In [ ]:
# # ==== PLOT TRAINING HISTORY ====
# plt.figure(figsize=(15, 5))

# # --- 1. Training & Validation Loss ---
# plt.subplot(1, 2, 1)
# plt.plot(training_history['train_loss'], label='Train Loss', color='blue')
# if CO_ORDINATES:
#     plt.plot(training_history['val_loss'], label='Val Loss', color='red')
# plt.title('Training Loss' if CO_ORDINATES else 'Loss / Accuracy')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.legend()
# plt.grid(True)

# # --- 2. Accuracy (only for classification) ---
# if not CO_ORDINATES:
#     plt.subplot(1, 2, 2)
#     plt.plot(training_history['train_acc'], label='Train Accuracy', color='blue')
#     plt.plot(training_history['val_acc'], label='Validation Accuracy', color='red')
#     plt.plot(training_history['val_top3_acc'], label='Validation Top-3 Accuracy', color='green')
#     plt.title('Accuracy Metrics')
#     plt.xlabel('Epoch')
#     plt.ylabel('Accuracy')
#     plt.legend()
#     plt.grid(True)

# plt.tight_layout()
# plt.show()

# # ==== PRINT FINAL METRICS ====
# if CO_ORDINATES:
#     print(f"\nFinal Validation Loss (MSE): {training_history['val_loss'][-1]:.4f}")
# else:
#     print(f"\nFinal Validation Accuracy: {training_history['val_acc'][-1]:.4f}")
#     print(f"Final Validation Top-3 Accuracy: {training_history['val_top3_acc'][-1]:.4f}")
